# UPLIFT MODEL SCRIPT (T-LEARNER WITH XGBOOST)
Reads from Excel `.xlsx`  
Outcome: `outcome_ed_90d`  
Treatment: `intervention_flag`


---
## 1. Install packages if needed
---


In [1]:
from pathlib import Path
import importlib.util
import sys

required_imports = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'xgboost': 'xgboost',
    'openpyxl': 'openpyxl',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
}
missing = [pip_name for import_name, pip_name in required_imports.items() if importlib.util.find_spec(import_name) is None]
if missing:
    raise ImportError('Install missing packages with: pip install ' + ' '.join(missing))

for candidate in [Path.cwd(), Path.cwd() / 'Code']:
    if (candidate / '_prism_model_utils.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find _prism_model_utils.py in the notebook folder or ./Code')


---
## 2. Load packages
---


In [2]:
from itertools import product
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb

try:
    import shap
except ImportError:
    shap = None

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNetCV, LogisticRegressionCV
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from _prism_model_utils import (
    GITHUB_XLSX_URL,
    align_to_columns,
    assert_xgb_booster_uses_cuda,
    clean_names_simple,
    ensure_output_folder,
    impute_categorical,
    impute_numeric,
    make_design_matrix,
    ntile_desc,
    project_root,
    read_prism_excel,
    require_columns,
    resolve_xgb_gpu_params,
    shap_importance_frame,
    split_train_test,
    to_binary,
    xgb_importance_frame,
    xgb_training_params,
)

warnings.filterwarnings('ignore', category=ConvergenceWarning)
PROJECT_ROOT = project_root()


---
## 3. FILE PATHS
---


In [3]:
# Raw GitHub URL to the Excel file
github_xlsx_url = GITHUB_XLSX_URL

# Output paths: keep T-learner and X-learner artifacts separate.
output_folder = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python')
tlearner_root_folder = ensure_output_folder(output_folder / 'T-Learner')
xgboost_output_folder = ensure_output_folder(tlearner_root_folder / 'XGBoost')
glmnet_output_folder = ensure_output_folder(tlearner_root_folder / 'GLMNet')

xgboost_output_path = xgboost_output_folder / 'uplift_scored_output.csv'
xgboost_summary_path = xgboost_output_folder / 'uplift_decile_summary.csv'
glmnet_output_path = glmnet_output_folder / 'uplift_scored_output.csv'
glmnet_summary_path = glmnet_output_folder / 'uplift_decile_summary.csv'

# Backward-compatible aliases for the primary XGBoost output.
output_path = xgboost_output_path
summary_path = xgboost_summary_path

print('Imported data from:', github_xlsx_url, '\n')
print('Project root resolved to:', PROJECT_ROOT, '\n')
print('Python output root:', output_folder, '\n')
print('XGBoost outputs will be saved to:', xgboost_output_folder, '\n')
print('GLMNET outputs will be saved to:', glmnet_output_folder, '\n')


Imported data from: https://raw.githubusercontent.com/ndesai777777/prism_repo/main/DataSets/PRP_1000_full_pretreatment.xlsx 

Project root resolved to: /home/sagemaker-user/prism_repo 

Python output root: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python 

XGBoost outputs will be saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost 

GLMNET outputs will be saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet 



---
## 4. HELPER FUNCTIONS
---


In [4]:
def safe_as_date(values):
    return pd.to_datetime(values, errors='coerce')


def present_columns(columns, df):
    return [column for column in columns if column in df.columns]


def safe_auc(y_true, y_pred, label):
    y_series = pd.Series(y_true).dropna()
    if y_series.nunique() < 2:
        print(f'{label} AUC: cannot calculate because only one outcome class is present')
        return np.nan
    auc_value = roc_auc_score(y_true, y_pred)
    print(f'{label} AUC: {auc_value:.4f}')
    return auc_value


REQUIRE_GPU_FOR_XGBOOST = True
RUN_CPU_ONLY_COMPARISON_MODELS = True
XGBOOST_CUDA_DEVICE = 0
XGB_GPU_PARAMS = resolve_xgb_gpu_params(cuda_device=XGBOOST_CUDA_DEVICE) if REQUIRE_GPU_FOR_XGBOOST else {}
print('XGBoost GPU required:', REQUIRE_GPU_FOR_XGBOOST)
print('XGBoost CUDA params used for training:', XGB_GPU_PARAMS)
print('CPU-only GLMNET comparison enabled:', RUN_CPU_ONLY_COMPARISON_MODELS)


def make_dmatrix(x_matrix, y=None):
    if y is None:
        return xgb.DMatrix(x_matrix, feature_names=list(x_matrix.columns))
    return xgb.DMatrix(x_matrix, label=np.asarray(y, dtype=float), feature_names=list(x_matrix.columns))


def fit_xgb_cv_grid(x_matrix, y, grid, nrounds_max=500, nfold=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    dtrain = make_dmatrix(x_matrix, y_array)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfold, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for XGBoost CV.')

    results = []
    best_model_info = None
    best_auc = -np.inf

    for params_grid in grid:
        params_i = xgb_training_params(
            XGB_GPU_PARAMS,
            {
                'max_depth': params_grid['max_depth'],
                'eta': params_grid['eta'],
                'min_child_weight': params_grid['min_child_weight'],
                'subsample': 0.8,
                'colsample_bytree': 0.8,
            },
            eval_metric='auc',
            seed=seed,
        )
        cv_i = xgb.cv(
            params=params_i,
            dtrain=dtrain,
            num_boost_round=nrounds_max,
            nfold=folds,
            stratified=True,
            early_stopping_rounds=20,
            seed=seed,
            verbose_eval=False,
        )
        auc_column = 'test-auc-mean'
        best_iter_i = int(cv_i[auc_column].idxmax())
        best_auc_i = float(cv_i.loc[best_iter_i, auc_column])
        best_nrounds_i = best_iter_i + 1
        results.append({
            'max_depth': params_grid['max_depth'],
            'eta': params_grid['eta'],
            'min_child_weight': params_grid['min_child_weight'],
            'best_nrounds': best_nrounds_i,
            'cv_auc': best_auc_i,
        })
        if best_auc_i > best_auc:
            best_auc = best_auc_i
            best_model_info = {'params': params_i, 'best_nrounds': best_nrounds_i, 'cv_auc': best_auc_i}

    search_results = pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True)
    final_model = xgb.train(
        params=best_model_info['params'],
        dtrain=dtrain,
        num_boost_round=best_model_info['best_nrounds'],
        verbose_eval=False,
    )
    return {
        'model': final_model,
        'best_params': best_model_info['params'],
        'best_nrounds': best_model_info['best_nrounds'],
        'best_cv_auc': best_model_info['cv_auc'],
        'search_results': search_results,
    }


class PrefitScaledLogisticPipeline:
    def __init__(self, scaler, model):
        self.named_steps = {
            'standardscaler': scaler,
            'logisticregressioncv': model,
        }

    def predict_proba(self, x_matrix):
        return self.named_steps['logisticregressioncv'].predict_proba(
            self.named_steps['standardscaler'].transform(x_matrix)
        )

    def predict(self, x_matrix):
        return self.named_steps['logisticregressioncv'].predict(
            self.named_steps['standardscaler'].transform(x_matrix)
        )


def fit_elastic_net(x_matrix, y, alpha_grid=np.round(np.arange(0, 1.01, 0.1), 1), nfolds=5, seed=123, prefit_scaler=None):
    y_array = np.asarray(y, dtype=float)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfolds, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for elastic-net CV.')

    if not RUN_CPU_ONLY_COMPARISON_MODELS:
        raise RuntimeError(
            'fit_elastic_net uses sklearn LogisticRegressionCV, which trains on CPU. '
            'Set RUN_CPU_ONLY_COMPARISON_MODELS = True to run this CPU comparison model.'
        )

    results = []
    best_pipeline = None
    best_auc = -np.inf
    best_alpha = np.nan
    best_lambda = np.nan
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    for alpha in alpha_grid:
        penalty = 'l2' if alpha == 0 else 'elasticnet'
        l1_ratios = None if alpha == 0 else [float(alpha)]
        cv_model = LogisticRegressionCV(
            Cs=np.logspace(-4, 4, 30),
            cv=cv,
            penalty=penalty,
            solver='saga',
            l1_ratios=l1_ratios,
            scoring='roc_auc',
            max_iter=10000,
            random_state=seed,
            refit=True,
        )
        if prefit_scaler is None:
            pipeline = make_pipeline(StandardScaler(), cv_model)
            pipeline.fit(x_matrix, y_array)
            fitted = pipeline.named_steps['logisticregressioncv']
        else:
            x_scaled = prefit_scaler.transform(x_matrix)
            cv_model.fit(x_scaled, y_array)
            fitted = cv_model
            pipeline = PrefitScaledLogisticPipeline(prefit_scaler, fitted)
        scores = fitted.scores_[1.0]
        auc_cv = float(np.nanmax(np.nanmean(scores, axis=0)))
        lambda_value = float(1 / fitted.C_[0])
        results.append({'alpha': float(alpha), 'lambda': lambda_value, 'cv_auc': auc_cv})
        if auc_cv > best_auc:
            best_auc = auc_cv
            best_alpha = float(alpha)
            best_lambda = lambda_value
            best_pipeline = pipeline

    return {
        'best_model': best_pipeline,
        'best_alpha': best_alpha,
        'best_lambda': best_lambda,
        'best_auc': best_auc,
        'search_results': pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True),
    }


def build_uplift_results(base_df, pred_treated, pred_control):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['uplift_bad_outcome'] = results['pred_ed_if_treated'] - results['pred_ed_if_control']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    return results


def summarize_uplift_deciles(results):
    return (
        results
        .groupby('uplift_decile', as_index=False)
        .agg(
            n=('outcome_ed_90d', 'size'),
            avg_benefit_score=('benefit_score', 'mean'),
            observed_ed_rate=('outcome_ed_90d', 'mean'),
            treated_pct=('intervention_flag', 'mean'),
            avg_pred_ed_if_treated=('pred_ed_if_treated', 'mean'),
            avg_pred_ed_if_control=('pred_ed_if_control', 'mean'),
        )
        .sort_values('uplift_decile')
    )


def print_highest_benefit(results, label, n=20):
    print(f'{label} top {n} highest-benefit members:')
    display(
        results
        .sort_values('benefit_score', ascending=False)
        [[
            'outcome_ed_90d',
            'intervention_flag',
            'pred_ed_if_treated',
            'pred_ed_if_control',
            'benefit_score',
            'uplift_decile',
        ]]
        .head(n)
    )
    print()


def glmnet_contribution_importance_frame(model_info, x_matrix, label):
    pipeline = model_info['best_model']
    scaler = pipeline.named_steps['standardscaler']
    glmnet_model = pipeline.named_steps['logisticregressioncv']
    x_scaled = scaler.transform(x_matrix)
    coefficients = glmnet_model.coef_.ravel()
    contributions = x_scaled * coefficients
    return (
        pd.DataFrame(
            {
                'feature': list(x_matrix.columns),
                'mean_abs_model_contribution': np.abs(contributions).mean(axis=0),
                'coefficient': coefficients,
                'model': label,
                'importance_type': 'standardized_logit_contribution',
            }
        )
        .sort_values('mean_abs_model_contribution', ascending=False)
        .reset_index(drop=True)
    )



NVIDIA GPU(s) detected by nvidia-smi:
  - 0, NVIDIA A10G, 595.71.05, 23028 MiB


XGBoost GPU training enabled with params: {'tree_method': 'hist', 'device': 'cuda:0'}
XGBoost GPU required: True
XGBoost CUDA params used for training: {'tree_method': 'hist', 'device': 'cuda:0'}
CPU-only GLMNET comparison enabled: True


---
## 5. READ EXCEL FILE
---


In [5]:
df_raw = read_prism_excel(github_xlsx_url)
df_raw.columns = clean_names_simple(df_raw.columns)

print('Rows:', len(df_raw))
print('Columns:', len(df_raw.columns))
print()
print('Column names after cleaning:')
print(list(df_raw.columns))
print()


Imported data from:
https://raw.githubusercontent.com/ndesai777777/prism_repo/main/DataSets/PRP_1000_full_pretreatment.xlsx



Rows: 1000
Columns: 44

Column names after cleaning:
['client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender', 'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'pregnancy_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier', 'intervention_flag', 'outcome_ed_90d']



---
## 6. CHECK REQUIRED COLUMNS
---


In [6]:
required_fields = ['outcome_ed_90d', 'intervention_flag']
require_columns(df_raw, required_fields)


---
## 7. BASIC CLEANUP
---


In [7]:
df = df_raw.copy()

for column in ['index_date', 'intervention_start_date', 'intervention_end_date']:
    if column in df.columns:
        df[column] = safe_as_date(df[column])

df['intervention_flag'] = to_binary(df['intervention_flag'])
df['outcome_ed_90d'] = to_binary(df['outcome_ed_90d'])


---
## 8. DERIVE DATE FEATURES
---


In [8]:
if 'intervention_start_date' in df.columns:
    df['intervention_start_month'] = df['intervention_start_date'].dt.month.astype(float)
    df['intervention_start_wday'] = (((df['intervention_start_date'].dt.dayofweek + 1) % 7) + 1).astype(float)
else:
    df['intervention_start_month'] = np.nan
    df['intervention_start_wday'] = np.nan

if {'index_date', 'intervention_start_date'}.issubset(df.columns):
    df['days_to_intervention_start'] = (df['intervention_start_date'] - df['index_date']).dt.days.astype(float)
else:
    df['days_to_intervention_start'] = np.nan

if {'intervention_start_date', 'intervention_end_date'}.issubset(df.columns):
    df['intervention_duration_calc'] = (df['intervention_end_date'] - df['intervention_start_date']).dt.days.astype(float)
else:
    df['intervention_duration_calc'] = np.nan

if 'intervention_days_active' not in df.columns:
    df['intervention_days_active'] = df['intervention_duration_calc']

# reverse for now
df = df_raw.copy()


---
## 9. SELECT PREDICTORS
---


In [9]:
candidate_predictors_all = [
    'client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender',
    'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag',
    'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag',
    'substance_use_flag', 'ckd_flag', 'pregnancy_flag', 'behavioral_health_risk_flag',
    'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag',
    'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m',
    'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag',
    'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier',
    'intervention_type', 'intervention_days_active', 'touches_per_month', 'outreach_attempts',
    'successful_contacts', 'avg_call_duration_min', 'max_call_duration_min', 'notes_escalation_flag',
    'community_referral_flag', 'pharmacy_review_flag', 'engagement_level',
    'days_to_intervention_start', 'intervention_start_month', 'intervention_start_wday',
]

candidate_predictors = [column for column in candidate_predictors_all if column in df.columns]
missing_predictors = [column for column in candidate_predictors_all if column not in df.columns]

if missing_predictors:
    print('Predictors not found in dataset:')
    print(missing_predictors)
else:
    print('All candidate predictors are present in dataset.')

model_df = df[['outcome_ed_90d', 'intervention_flag', *candidate_predictors]].copy()
model_df['outcome_ed_90d'] = to_binary(model_df['outcome_ed_90d'])
model_df['intervention_flag'] = to_binary(model_df['intervention_flag'])
model_df = model_df[model_df['outcome_ed_90d'].notna() & model_df['intervention_flag'].notna()].copy()

missing_in_model_df = [column for column in df.columns if column not in model_df.columns]
if missing_in_model_df:
    print('Columns in dataframe but not in model:')
    print(missing_in_model_df)
else:
    print('All dataframe columns are present in model.')
print('Modeling rows after dropping missing outcome/treatment:', len(model_df))
print()


Predictors not found in dataset:
['intervention_type', 'intervention_days_active', 'touches_per_month', 'outreach_attempts', 'successful_contacts', 'avg_call_duration_min', 'max_call_duration_min', 'notes_escalation_flag', 'community_referral_flag', 'pharmacy_review_flag', 'engagement_level', 'days_to_intervention_start', 'intervention_start_month', 'intervention_start_wday']
All dataframe columns are present in model.
Modeling rows after dropping missing outcome/treatment: 1000



---
## 10. DATA TYPE HANDLING
---


In [10]:
model_df.columns

Index(['outcome_ed_90d', 'intervention_flag', 'client_contract',
       'service_region', 'program', 'case_manager_name', 'age', 'gender',
       'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag',
       'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag',
       'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag',
       'pregnancy_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag',
       'housing_instability_flag', 'transportation_barrier_flag',
       'utilities_insecurity_flag', 'pcp_visits_last_6m',
       'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m',
       'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
       'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag',
       'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score',
       'percolator_clinical_score', 'percolator_sdoh_score',
       'current_risk_score', 'risk_tier'],
      dtype='object')

In [11]:
flag_like_cols = [column for column in model_df.columns if column.endswith('_flag')]
for column in flag_like_cols:
    model_df[column] = to_binary(model_df[column])

possible_numeric_cols = [
    'age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
    'rx_count_last_6m', 'med_adherence_pdc', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score',
    'intervention_days_active', 'touches_per_month', 'outreach_attempts', 'successful_contacts',
    'avg_call_duration_min', 'max_call_duration_min', 'days_to_intervention_start',
    'intervention_start_month', 'intervention_start_wday',
]

for column in present_columns(possible_numeric_cols, model_df):
    model_df[column] = pd.to_numeric(model_df[column], errors='coerce')

for column in model_df.columns:
    if column in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if not pd.api.types.is_numeric_dtype(model_df[column]):
        model_df[column] = impute_categorical(model_df[column])

for column in model_df.columns:
    if column in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if pd.api.types.is_numeric_dtype(model_df[column]):
        model_df[column] = impute_numeric(model_df[column])

unique_counts = model_df.apply(lambda column: column.dropna().nunique())
keep_cols = list(unique_counts[unique_counts > 1].index)
model_df = model_df.loc[:, keep_cols].copy().reset_index(drop=True)
model_df.insert(0, 'member_id', np.arange(len(model_df), dtype=int))

print('Final modeling columns:')
print(list(model_df.columns))
print()


Final modeling columns:
['member_id', 'outcome_ed_90d', 'intervention_flag', 'client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender', 'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier']



---
## 11. TRAIN / TEST SPLIT
---


In [12]:
train_df, test_df = split_train_test(
    model_df,
    train_fraction=0.70,
    seed=123,
    stratify_columns=['intervention_flag', 'outcome_ed_90d'],
)

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))
print()


Training rows: 700
Testing rows: 300



---
## 12. SEPARATE TREATED / CONTROL
---


In [13]:
train_treated = train_df[train_df['intervention_flag'] == 1].copy()
train_control = train_df[train_df['intervention_flag'] == 0].copy()

print('Training treated rows:', len(train_treated))
print('Training control rows:', len(train_control))
print()

if len(train_treated) < 50:
    raise ValueError('Too few treated rows to train a stable model.')
if len(train_control) < 50:
    raise ValueError('Too few control rows to train a stable model.')


Training treated rows: 276
Training control rows: 424



---
## 13. BUILD MODEL MATRICES
---


In [14]:
id_cols = ['member_id']
feature_cols = [
    column for column in model_df.columns
    if column not in ['outcome_ed_90d', 'intervention_flag', *id_cols]
]

train_treated_x_df = train_treated[feature_cols].copy()
train_control_x_df = train_control[feature_cols].copy()
test_x_df = test_df[feature_cols].copy()

combined_matrix, split_matrices = make_design_matrix([train_treated_x_df, train_control_x_df, test_x_df])
x_treated, x_control, x_test = split_matrices

y_treated = train_treated['outcome_ed_90d'].astype(float).to_numpy()
y_control = train_control['outcome_ed_90d'].astype(float).to_numpy()


---
## WRITE-UP DATA REVIEW SUMMARY
---


In [15]:
def is_binary_indicator_column(series):
    non_missing = pd.Series(series).dropna()
    if non_missing.empty:
        return False
    numeric_values = pd.to_numeric(non_missing, errors='coerce')
    if numeric_values.isna().any():
        return False
    return set(numeric_values.unique()).issubset({0, 1}) and numeric_values.nunique() <= 2


model_feature_type_rows = []
for column in feature_cols:
    series = model_df[column]
    if is_binary_indicator_column(series):
        predictor_type = 'Binary indicator'
    elif column in possible_numeric_cols or pd.api.types.is_numeric_dtype(series):
        predictor_type = 'Continuous/count numeric'
    else:
        predictor_type = 'Multi-level categorical'
    model_feature_type_rows.append({'variable': column, 'predictor_type': predictor_type})

model_feature_type_summary = pd.DataFrame(model_feature_type_rows)
predictor_type_counts = model_feature_type_summary['predictor_type'].value_counts().to_dict()

continuous_count_predictors = int(predictor_type_counts.get('Continuous/count numeric', 0))
binary_indicator_predictors = int(predictor_type_counts.get('Binary indicator', 0))
multilevel_categorical_predictors = int(predictor_type_counts.get('Multi-level categorical', 0))

data_review_summary = pd.DataFrame(
    [
        {
            'total_members': len(model_df),
            'treated_members': int((model_df['intervention_flag'] == 1).sum()),
            'control_members': int((model_df['intervention_flag'] == 0).sum()),
            'treatment_rate': float(model_df['intervention_flag'].mean()),
            'outcome_events': int((model_df['outcome_ed_90d'] == 1).sum()),
            'outcome_prevalence': float(model_df['outcome_ed_90d'].mean()),
            'treated_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 1, 'outcome_ed_90d'].mean()),
            'control_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 0, 'outcome_ed_90d'].mean()),
            'number_of_predictors': len(feature_cols),
            'number_of_continuous_count_predictors': continuous_count_predictors,
            'number_of_binary_indicator_predictors': binary_indicator_predictors,
            'number_of_multilevel_categorical_predictors': multilevel_categorical_predictors,
            'number_of_model_matrix_columns': x_test.shape[1],
        }
    ]
)

data_review_summary_path = output_folder / 'data_review_summary.csv'
data_review_summary.to_csv(data_review_summary_path, index=False)

print('Data review summary:')
display(data_review_summary)
print('Data review summary written to:', data_review_summary_path)


Data review summary:


,total_members,treated_members,control_members,treatment_rate,outcome_events,outcome_prevalence,treated_outcome_rate,control_outcome_rate,number_of_predictors,number_of_continuous_count_predictors,number_of_binary_indicator_predictors,number_of_multilevel_categorical_predictors,number_of_model_matrix_columns
0,1000,394,606,0.394,60,0.06,0.040609,0.072607,41,14,18,9,77


Data review summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/data_review_summary.csv


---
## WRITE-UP PREDICTOR DATA DICTIONARY AND SUMMARY
---


In [16]:
predictor_category_lookup = {
    # Demographics
    'client_contract': 'Demographics',
    'service_region': 'Demographics',
    'program': 'Demographics',
    'case_manager_name': 'Demographics',
    'age': 'Demographics',
    'gender': 'Demographics',
    'dual_eligible': 'Demographics',
    'county': 'Demographics',
    'plan_type': 'Demographics',
    'language': 'Demographics',
    'living_alone_flag': 'Demographics',

    # Clinical Conditions
    'diabetes_flag': 'Clinical Conditions',
    'chf_flag': 'Clinical Conditions',
    'copd_flag': 'Clinical Conditions',
    'asthma_flag': 'Clinical Conditions',
    'depression_flag': 'Clinical Conditions',
    'anxiety_flag': 'Clinical Conditions',
    'substance_use_flag': 'Clinical Conditions',
    'ckd_flag': 'Clinical Conditions',
    'pregnancy_flag': 'Clinical Conditions',
    'behavioral_health_risk_flag': 'Clinical Conditions',

    # SDOH
    'food_insecurity_flag': 'SDOH',
    'housing_instability_flag': 'SDOH',
    'transportation_barrier_flag': 'SDOH',
    'utilities_insecurity_flag': 'SDOH',

    # Utilization
    'pcp_visits_last_6m': 'Utilization',
    'specialist_visits_last_6m': 'Utilization',
    'ed_visits_last_30d': 'Utilization',
    'ed_visits_last_6m': 'Utilization',
    'admits_last_6m': 'Utilization',
    'observation_stays_last_6m': 'Utilization',

    # Pharmacy
    'total_cost_last_6m': 'Pharmacy',
    'rx_count_last_6m': 'Pharmacy',
    'med_adherence_pdc': 'Pharmacy',
    'high_cost_drug_flag': 'Pharmacy',
    'opioid_flag': 'Pharmacy',
    'polypharmacy_flag': 'Pharmacy',

    # Risk Scores
    'utilization_score': 'Risk Scores',
    'clinical_score': 'Risk Scores',
    'sdoh_score': 'Risk Scores',
    'percolator_utilization_score': 'Risk Scores',
    'percolator_clinical_score': 'Risk Scores',
    'percolator_sdoh_score': 'Risk Scores',
    'current_risk_score': 'Risk Scores',
    'risk_tier': 'Risk Scores',
}

predictor_description_lookup = {
    'client_contract': 'Client contract or line of business associated with the member.',
    'service_region': 'Service region where the member receives care management support.',
    'program': 'Care management program or operational program assignment.',
    'case_manager_name': 'Assigned case manager identifier/name.',
    'age': 'Member age at the index point.',
    'gender': 'Member gender category.',
    'dual_eligible': 'Indicator or category for Medicare/Medicaid dual eligibility.',
    'county': 'Member county of residence.',
    'plan_type': 'Health plan product or plan type.',
    'language': 'Preferred or primary language.',
    'living_alone_flag': 'Indicator that the member lives alone.',
    'diabetes_flag': 'Indicator for diabetes diagnosis/history.',
    'chf_flag': 'Indicator for congestive heart failure diagnosis/history.',
    'copd_flag': 'Indicator for COPD diagnosis/history.',
    'asthma_flag': 'Indicator for asthma diagnosis/history.',
    'depression_flag': 'Indicator for depression diagnosis/history.',
    'anxiety_flag': 'Indicator for anxiety diagnosis/history.',
    'substance_use_flag': 'Indicator for substance use-related risk or diagnosis.',
    'ckd_flag': 'Indicator for chronic kidney disease diagnosis/history.',
    'pregnancy_flag': 'Indicator for pregnancy status/history.',
    'behavioral_health_risk_flag': 'Indicator for behavioral health risk.',
    'food_insecurity_flag': 'Indicator for food insecurity.',
    'housing_instability_flag': 'Indicator for housing instability.',
    'transportation_barrier_flag': 'Indicator for transportation barriers.',
    'utilities_insecurity_flag': 'Indicator for utility insecurity.',
    'pcp_visits_last_6m': 'Number of primary care visits in the last 6 months.',
    'specialist_visits_last_6m': 'Number of specialist visits in the last 6 months.',
    'ed_visits_last_30d': 'Number of ED visits in the last 30 days.',
    'ed_visits_last_6m': 'Number of ED visits in the last 6 months.',
    'admits_last_6m': 'Number of inpatient admissions in the last 6 months.',
    'observation_stays_last_6m': 'Number of observation stays in the last 6 months.',
    'total_cost_last_6m': 'Total healthcare cost in the last 6 months.',
    'rx_count_last_6m': 'Number of prescriptions or medication fills in the last 6 months.',
    'med_adherence_pdc': 'Medication adherence proportion of days covered.',
    'high_cost_drug_flag': 'Indicator for high-cost drug use.',
    'opioid_flag': 'Indicator for opioid medication use.',
    'polypharmacy_flag': 'Indicator for polypharmacy risk.',
    'percolator_utilization_score': 'Composite utilization risk score.',
    'percolator_clinical_score': 'Composite clinical risk score.',
    'percolator_sdoh_score': 'Composite social determinants of health risk score.',
    'current_risk_score': 'Current overall risk score.',
    'risk_tier': 'Categorical risk tier assignment.',
    'intervention_type': 'Type of intervention assigned or delivered.',
    'intervention_days_active': 'Number of days the intervention was active.',
    'touches_per_month': 'Average number of care management touches per month.',
    'outreach_attempts': 'Number of outreach attempts.',
    'successful_contacts': 'Number of successful contacts.',
    'avg_call_duration_min': 'Average call duration in minutes.',
    'max_call_duration_min': 'Maximum call duration in minutes.',
    'notes_escalation_flag': 'Indicator for escalation noted in care management documentation.',
    'community_referral_flag': 'Indicator for community referral.',
    'pharmacy_review_flag': 'Indicator for pharmacy review.',
    'engagement_level': 'Member engagement level category.',
    'days_to_intervention_start': 'Days between index date and intervention start.',
    'intervention_start_month': 'Calendar month of intervention start.',
    'intervention_start_wday': 'Day of week of intervention start.',
}


def predictor_category(column):
    return predictor_category_lookup.get(column, 'Other / derived')


def predictor_description(column):
    return predictor_description_lookup.get(column, f'Model predictor derived from `{column}`.')


def compact_examples(series, max_items=5):
    values = series.dropna().astype(str).drop_duplicates().head(max_items).tolist()
    return '; '.join(values)


def top_value_summary(series, max_items=5):
    counts = series.dropna().astype(str).value_counts().head(max_items)
    return '; '.join(f'{idx} ({count})' for idx, count in counts.items())


def is_binary_predictor(column, series):
    if column not in feature_cols:
        return False
    non_missing = pd.Series(series).dropna()
    if non_missing.empty:
        return False
    numeric_values = pd.to_numeric(non_missing, errors='coerce')
    if numeric_values.isna().any():
        return False
    return set(numeric_values.unique()).issubset({0, 1}) and numeric_values.nunique() <= 2


def predictor_type(column, series):
    if is_binary_predictor(column, series):
        return 'Binary indicator'
    if column in possible_numeric_cols:
        return 'Continuous/count numeric'
    return 'Multi-level categorical'


predictor_distribution_folder = ensure_output_folder(output_folder / 'Predictor_Distributions')
numeric_distribution_folder = ensure_output_folder(predictor_distribution_folder / 'Numeric')
categorical_distribution_folder = ensure_output_folder(predictor_distribution_folder / 'Categorical')

predictor_dictionary_rows = []
for column in candidate_predictors:
    raw_series = df[column] if column in df.columns else pd.Series(dtype='object')
    model_series = model_df[column] if column in model_df.columns else raw_series
    numeric_values = pd.to_numeric(raw_series, errors='coerce')
    is_numeric_like = pd.api.types.is_numeric_dtype(model_series) or numeric_values.notna().sum() > 0
    model_matrix_matches = [matrix_col for matrix_col in x_test.columns if matrix_col == column or matrix_col.startswith(f'{column}_')]
    predictor_dictionary_rows.append(
        {
            'variable': column,
            'category': predictor_category(column),
            'data_type': str(raw_series.dtype),
            'model_ready_data_type': str(model_series.dtype),
            'description': predictor_description(column),
            'predictor_type': predictor_type(column, model_series),
            'included_in_model': column in feature_cols,
            'included_in_model_matrix': len(model_matrix_matches) > 0,
            'model_matrix_columns': '; '.join(model_matrix_matches[:20]),
            'missing_count': int(raw_series.isna().sum()),
            'missing_pct': float(raw_series.isna().mean()),
            'unique_values': int(raw_series.dropna().nunique()),
            'example_values': compact_examples(raw_series),
            'min_value': float(numeric_values.min()) if is_numeric_like and numeric_values.notna().any() else np.nan,
            'max_value': float(numeric_values.max()) if is_numeric_like and numeric_values.notna().any() else np.nan,
        }
    )

predictor_data_dictionary = pd.DataFrame(predictor_dictionary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
predictor_data_dictionary_path = predictor_distribution_folder / 'predictor_data_dictionary.csv'
predictor_data_dictionary.to_csv(predictor_data_dictionary_path, index=False)

numeric_summary_rows = []
categorical_summary_rows = []
for column in candidate_predictors:
    raw_series = df[column] if column in df.columns else pd.Series(dtype='object')
    numeric_values = pd.to_numeric(raw_series, errors='coerce')
    numeric_share = numeric_values.notna().mean() if len(raw_series) else 0
    treat_as_numeric = (
        column in possible_numeric_cols
        and not is_binary_predictor(column, raw_series)
        and numeric_values.notna().any()
    )

    if treat_as_numeric:
        numeric_summary_rows.append(
            {
                'variable': column,
                'category': predictor_category(column),
                'n': int(numeric_values.notna().sum()),
                'missing_count': int(raw_series.isna().sum()),
                'missing_pct': float(raw_series.isna().mean()),
                'mean': float(numeric_values.mean()),
                'median': float(numeric_values.median()),
                'std': float(numeric_values.std()),
                'min': float(numeric_values.min()),
                'p25': float(numeric_values.quantile(0.25)),
                'p75': float(numeric_values.quantile(0.75)),
                'max': float(numeric_values.max()),
            }
        )
    else:
        non_missing = raw_series.dropna().astype(str)
        value_counts = non_missing.value_counts()
        mode_value = value_counts.index[0] if len(value_counts) else np.nan
        mode_count = int(value_counts.iloc[0]) if len(value_counts) else 0
        categorical_summary_rows.append(
            {
                'variable': column,
                'category': predictor_category(column),
                'n': int(non_missing.shape[0]),
                'missing_count': int(raw_series.isna().sum()),
                'missing_pct': float(raw_series.isna().mean()),
                'unique_values': int(non_missing.nunique()),
                'mode': mode_value,
                'mode_count': mode_count,
                'mode_pct': float(mode_count / len(raw_series)) if len(raw_series) else np.nan,
                'top_values': top_value_summary(raw_series),
            }
        )

numeric_predictor_summary = pd.DataFrame(numeric_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
categorical_predictor_summary = pd.DataFrame(categorical_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)

numeric_predictor_summary_path = predictor_distribution_folder / 'numeric_predictor_summary.csv'
categorical_predictor_summary_path = predictor_distribution_folder / 'categorical_predictor_summary.csv'
numeric_predictor_summary.to_csv(numeric_predictor_summary_path, index=False)
categorical_predictor_summary.to_csv(categorical_predictor_summary_path, index=False)

print('Predictor data dictionary:')
display(predictor_data_dictionary)
print('Predictor data dictionary written to:', predictor_data_dictionary_path)
print()

print('Numeric predictor summary:')
display(numeric_predictor_summary)
print('Numeric predictor summary written to:', numeric_predictor_summary_path)
print()

print('Categorical predictor summary:')
display(categorical_predictor_summary)
print('Categorical predictor summary written to:', categorical_predictor_summary_path)



Predictor data dictionary:


,variable,category,data_type,model_ready_data_type,description,predictor_type,included_in_model,included_in_model_matrix,model_matrix_columns,missing_count,missing_pct,unique_values,example_values,min_value,max_value
0,anxiety_flag,Clinical Conditions,int64,float64,Indicator for anxiety diagnosis/history.,Binary indicator,True,True,anxiety_flag,0,0.0,2,0; 1,0.000,1.000
1,asthma_flag,Clinical Conditions,int64,float64,Indicator for asthma diagnosis/history.,Binary indicator,True,True,asthma_flag,0,0.0,2,1; 0,0.000,1.000
2,behavioral_health_risk_flag,Clinical Conditions,int64,float64,Indicator for behavioral health risk.,Binary indicator,True,True,behavioral_health_risk_flag,0,0.0,2,0; 1,0.000,1.000
3,chf_flag,Clinical Conditions,int64,float64,Indicator for congestive heart failure diagnos...,Binary indicator,True,True,chf_flag,0,0.0,2,0; 1,0.000,1.000
4,ckd_flag,Clinical Conditions,int64,float64,Indicator for chronic kidney disease diagnosis...,Binary indicator,True,True,ckd_flag,0,0.0,2,0; 1,0.000,1.000
5,copd_flag,Clinical Conditions,int64,float64,Indicator for COPD diagnosis/history.,Binary indicator,True,True,copd_flag,0,0.0,2,0; 1,0.000,1.000
6,depression_flag,Clinical Conditions,int64,float64,Indicator for depression diagnosis/history.,Binary indicator,True,True,depression_flag,0,0.0,2,0; 1,0.000,1.000
7,diabetes_flag,Clinical Conditions,int64,float64,Indicator for diabetes diagnosis/history.,Binary indicator,True,True,diabetes_flag,0,0.0,2,0; 1,0.000,1.000
8,pregnancy_flag,Clinical Conditions,int64,int64,Indicator for pregnancy status/history.,Multi-level categorical,False,False,,0,0.0,1,0,0.000,0.000
9,substance_use_flag,Clinical Conditions,int64,float64,Indicator for substance use-related risk or di...,Binary indicator,True,True,substance_use_flag,0,0.0,2,0; 1,0.000,1.000


Predictor data dictionary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/predictor_data_dictionary.csv

Numeric predictor summary:


,variable,category,n,missing_count,missing_pct,mean,median,std,min,p25,p75,max
0,age,Demographics,1000,0,0.0,57.354000,57.0000,14.779212,18.000,48.000,68.00000,90.000
1,med_adherence_pdc,Pharmacy,1000,0,0.0,0.757421,0.7775,0.137761,0.238,0.680,0.85725,0.997
2,rx_count_last_6m,Pharmacy,1000,0,0.0,6.200000,6.0000,3.521704,0.000,4.000,8.00000,20.000
3,total_cost_last_6m,Pharmacy,1000,0,0.0,4102.298990,3658.5700,1898.596493,1092.950,2690.125,5134.61500,12520.110
4,current_risk_score,Risk Scores,1000,0,0.0,43.618300,42.8000,9.458610,14.500,37.100,49.40000,79.800
5,percolator_clinical_score,Risk Scores,1000,0,0.0,46.240000,44.7000,13.783831,12.700,36.600,54.12500,96.500
6,percolator_sdoh_score,Risk Scores,1000,0,0.0,35.523200,33.9000,13.644978,2.900,25.500,43.82500,85.000
7,percolator_utilization_score,Risk Scores,1000,0,0.0,45.172800,44.0000,12.878598,13.900,36.400,52.40000,100.000
8,admits_last_6m,Utilization,1000,0,0.0,0.339000,0.0000,0.613560,0.000,0.000,1.00000,4.000
9,ed_visits_last_30d,Utilization,1000,0,0.0,0.222000,0.0000,0.474279,0.000,0.000,0.00000,2.000


Numeric predictor summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/numeric_predictor_summary.csv

Categorical predictor summary:


,variable,category,n,missing_count,missing_pct,unique_values,mode,mode_count,mode_pct,top_values
0,anxiety_flag,Clinical Conditions,1000,0,0.0,2,0,727,0.727,0 (727); 1 (273)
1,asthma_flag,Clinical Conditions,1000,0,0.0,2,0,832,0.832,0 (832); 1 (168)
2,behavioral_health_risk_flag,Clinical Conditions,1000,0,0.0,2,0,593,0.593,0 (593); 1 (407)
3,chf_flag,Clinical Conditions,1000,0,0.0,2,0,851,0.851,0 (851); 1 (149)
4,ckd_flag,Clinical Conditions,1000,0,0.0,2,0,847,0.847,0 (847); 1 (153)
5,copd_flag,Clinical Conditions,1000,0,0.0,2,0,826,0.826,0 (826); 1 (174)
6,depression_flag,Clinical Conditions,1000,0,0.0,2,0,713,0.713,0 (713); 1 (287)
7,diabetes_flag,Clinical Conditions,1000,0,0.0,2,0,686,0.686,0 (686); 1 (314)
8,pregnancy_flag,Clinical Conditions,1000,0,0.0,1,0,1000,1.000,0 (1000)
9,substance_use_flag,Clinical Conditions,1000,0,0.0,2,0,840,0.840,0 (840); 1 (160)


Categorical predictor summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/categorical_predictor_summary.csv


---
## PREDICTOR DISTRIBUTION VISUALS
---


In [17]:
from matplotlib.backends.backend_pdf import PdfPages
import re


numeric_distribution_pdf_path = predictor_distribution_folder / 'numeric_predictor_distributions.pdf'
categorical_distribution_pdf_path = predictor_distribution_folder / 'categorical_predictor_distributions.pdf'
distribution_visual_index_path = predictor_distribution_folder / 'predictor_distribution_visual_index.csv'

sns.set_theme(style='whitegrid')


def safe_filename(value):
    safe_value = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')
    return safe_value or 'unnamed'


def plot_numeric_distribution(column, folder):
    values = pd.to_numeric(df[column], errors='coerce').dropna()
    if values.empty:
        return None

    fig, ax = plt.subplots(figsize=(8, 4.8))
    unique_count = values.nunique()
    if unique_count <= 15:
        sns.histplot(values, discrete=True, shrink=0.8, ax=ax, color='#4c78a8')
    else:
        sns.histplot(values, bins=30, kde=True, ax=ax, color='#4c78a8')
    ax.set_title(f'{column} Distribution')
    ax.set_xlabel(column)
    ax.set_ylabel('Member count')
    fig.tight_layout()

    output_path = folder / f'{safe_filename(column)}_histogram.png'
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, output_path


def plot_categorical_distribution(column, folder, top_n=20):
    values = df[column].dropna().astype(str)
    if values.empty:
        return None

    counts = values.value_counts()
    if len(counts) > top_n:
        counts = pd.concat([counts.head(top_n), pd.Series({'Other': counts.iloc[top_n:].sum()})])

    fig_height = max(4.8, min(12, 0.35 * len(counts) + 2.2))
    fig, ax = plt.subplots(figsize=(9, fig_height))
    plot_df = counts.reset_index()
    plot_df.columns = [column, 'member_count']
    sns.barplot(data=plot_df, x='member_count', y=column, ax=ax, color='#59a14f')
    ax.set_title(f'{column} Distribution')
    ax.set_xlabel('Member count')
    ax.set_ylabel(column)
    for patch in ax.patches:
        width = patch.get_width()
        if pd.notna(width):
            ax.annotate(
                f'{int(width):,}',
                (width, patch.get_y() + patch.get_height() / 2),
                ha='left',
                va='center',
                fontsize=8,
                xytext=(4, 0),
                textcoords='offset points',
            )
    fig.tight_layout()

    output_path = folder / f'{safe_filename(column)}_bar_chart.png'
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, output_path


distribution_visual_rows = []

numeric_columns_for_plots = numeric_predictor_summary['variable'].tolist() if not numeric_predictor_summary.empty else []
with PdfPages(numeric_distribution_pdf_path) as pdf:
    for column in numeric_columns_for_plots:
        result = plot_numeric_distribution(column, numeric_distribution_folder)
        if result is None:
            continue
        fig, output_path = result
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        distribution_visual_rows.append({
            'variable': column,
            'category': predictor_category(column),
            'visual_type': 'numeric_histogram',
            'output_file': str(output_path),
        })

categorical_columns_for_plots = categorical_predictor_summary['variable'].tolist() if not categorical_predictor_summary.empty else []
with PdfPages(categorical_distribution_pdf_path) as pdf:
    for column in categorical_columns_for_plots:
        result = plot_categorical_distribution(column, categorical_distribution_folder)
        if result is None:
            continue
        fig, output_path = result
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        distribution_visual_rows.append({
            'variable': column,
            'category': predictor_category(column),
            'visual_type': 'categorical_bar_chart',
            'output_file': str(output_path),
        })

distribution_visual_index = pd.DataFrame(distribution_visual_rows)
distribution_visual_index.to_csv(distribution_visual_index_path, index=False)

print('Predictor distribution visuals written to:', predictor_distribution_folder)
print('Numeric predictor distribution PDF:', numeric_distribution_pdf_path)
print('Categorical predictor distribution PDF:', categorical_distribution_pdf_path)
print('Distribution visual index written to:', distribution_visual_index_path)
display(distribution_visual_index)


Predictor distribution visuals written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions
Numeric predictor distribution PDF: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/numeric_predictor_distributions.pdf
Categorical predictor distribution PDF: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/categorical_predictor_distributions.pdf
Distribution visual index written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/predictor_distribution_visual_index.csv


,variable,category,visual_type,output_file
0,age,Demographics,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
1,med_adherence_pdc,Pharmacy,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
2,rx_count_last_6m,Pharmacy,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
3,total_cost_last_6m,Pharmacy,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
4,current_risk_score,Risk Scores,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
5,percolator_clinical_score,Risk Scores,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
6,percolator_sdoh_score,Risk Scores,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
7,percolator_utilization_score,Risk Scores,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
8,admits_last_6m,Utilization,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...
9,ed_visits_last_30d,Utilization,numeric_histogram,/home/sagemaker-user/prism_repo/Outputs/Uplift...


---
## 14. TRAIN XGBOOST MODELS
---


In [18]:
print('Unique y_treated values:')
print(np.sort(pd.unique(y_treated)))
print()

print('Unique y_control values:')
print(np.sort(pd.unique(y_control)))
print()

if not set(pd.Series(y_treated).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_treated contains values other than 0 and 1.')
if not set(pd.Series(y_control).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_control contains values other than 0 and 1.')

dtrain_treated = make_dmatrix(x_treated, y_treated)
dtrain_control = make_dmatrix(x_control, y_control)

params = xgb_training_params(
    XGB_GPU_PARAMS,
    {
        'max_depth': 4,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    },
    eval_metric='logloss',
    seed=123,
)

model_treated = xgb.train(params=params, dtrain=dtrain_treated, num_boost_round=150, verbose_eval=False)
model_control = xgb.train(params=params, dtrain=dtrain_control, num_boost_round=150, verbose_eval=False)
assert_xgb_booster_uses_cuda(model_treated, 'Baseline treated XGBoost model')
assert_xgb_booster_uses_cuda(model_control, 'Baseline control XGBoost model')

print('Models trained successfully with XGBoost GPU params:', XGB_GPU_PARAMS)
print()

test_treated_pos = np.where(test_df['intervention_flag'].to_numpy() == 1)[0]
test_control_pos = np.where(test_df['intervention_flag'].to_numpy() == 0)[0]

pred_treated = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated, 'XGBoost Treated model')

pred_control = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control, 'XGBoost Control model')


Unique y_treated values:
[0. 1.]

Unique y_control values:
[0. 1.]



Baseline treated XGBoost model booster config confirms CUDA/GPU training.
Baseline control XGBoost model booster config confirms CUDA/GPU training.
Models trained successfully with XGBoost GPU params: {'tree_method': 'hist', 'device': 'cuda:0'}

XGBoost Treated model AUC: 0.6796
XGBoost Control model AUC: 0.4306


---
## XGBOOST CV GRID SEARCH FUNCTION
---


In [19]:
xgb_grid = [
    {'max_depth': max_depth, 'eta': eta, 'min_child_weight': min_child_weight}
    for max_depth, eta, min_child_weight in product([3, 4, 5], [0.03, 0.05, 0.10], [1, 5])
]


---
## TRAIN TREATED MODEL WITH CV GRID SEARCH
---


In [20]:
xgb_treated_cv = fit_xgb_cv_grid(x_matrix=x_treated, y=y_treated, grid=xgb_grid, nrounds_max=500, nfold=5)
model_treated = xgb_treated_cv['model']
assert_xgb_booster_uses_cuda(model_treated, 'CV-tuned treated XGBoost model')

print('XGBoost Treated best CV AUC:', round(xgb_treated_cv['best_cv_auc'], 4))
print('XGBoost Treated best nrounds:', xgb_treated_cv['best_nrounds'])
print('XGBoost Treated best params:')
print(xgb_treated_cv['best_params'])


CV-tuned treated XGBoost model booster config confirms CUDA/GPU training.
XGBoost Treated best CV AUC: 0.7399
XGBoost Treated best nrounds: 8
XGBoost Treated best params:
{'objective': 'binary:logistic', 'eval_metric': 'auc', 'seed': 123, 'verbosity': 0, 'max_depth': 3, 'eta': 0.03, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist', 'device': 'cuda:0'}


---
## TRAIN CONTROL MODEL WITH CV GRID SEARCH
---


In [21]:
xgb_control_cv = fit_xgb_cv_grid(x_matrix=x_control, y=y_control, grid=xgb_grid, nrounds_max=500, nfold=5)
model_control = xgb_control_cv['model']
assert_xgb_booster_uses_cuda(model_control, 'CV-tuned control XGBoost model')

print('XGBoost Control best CV AUC:', round(xgb_control_cv['best_cv_auc'], 4))
print('XGBoost Control best nrounds:', xgb_control_cv['best_nrounds'])
print('XGBoost Control best params:')
print(xgb_control_cv['best_params'])


CV-tuned control XGBoost model booster config confirms CUDA/GPU training.
XGBoost Control best CV AUC: 0.668
XGBoost Control best nrounds: 15
XGBoost Control best params:
{'objective': 'binary:logistic', 'eval_metric': 'auc', 'seed': 123, 'verbosity': 0, 'max_depth': 3, 'eta': 0.05, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist', 'device': 'cuda:0'}


---
## TEST AUC FOR CV-TUNED XGBOOST MODELS
---


In [22]:
pred_treated_cv_xgb = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_xgb, 'XGBoost Treated CV-tuned test')

pred_control_cv_xgb = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_xgb, 'XGBoost Control CV-tuned test')

print('CV-tuned XGBoost models trained successfully.')
print()


XGBoost Treated CV-tuned test AUC: 0.8354
XGBoost Control CV-tuned test AUC: 0.5833
CV-tuned XGBoost models trained successfully.



---
## CPU-ONLY GLMNET COMPARISON
---


In [23]:
if RUN_CPU_ONLY_COMPARISON_MODELS:
    print('Training GLMNET comparison models on CPU with sklearn LogisticRegressionCV.')
    print()
    glmnet_shared_scaler = StandardScaler().fit(pd.concat([x_treated, x_control], axis=0))
    print('GLMNET shared scaler fit on combined treated/control training matrix.')
    enet_treated = fit_elastic_net(x_treated, y_treated, prefit_scaler=glmnet_shared_scaler)

    print('Best treated alpha:', enet_treated['best_alpha'])
    print('Best treated lambda:', enet_treated['best_lambda'])
    print('Best treated CV AUC:', round(enet_treated['best_auc'], 4))
    print()

    enet_control = fit_elastic_net(x_control, y_control, prefit_scaler=glmnet_shared_scaler)

    print('Best control alpha:', enet_control['best_alpha'])
    print('Best control lambda:', enet_control['best_lambda'])
    print('Best control CV AUC:', round(enet_control['best_auc'], 4))
    print()
else:
    glmnet_shared_scaler = None
    enet_treated = None
    enet_control = None
    print('Skipped GLMNET comparison models because RUN_CPU_ONLY_COMPARISON_MODELS is False.')
    print()


Training GLMNET comparison models on CPU with sklearn LogisticRegressionCV.

GLMNET shared scaler fit on combined treated/control training matrix.


Best treated alpha: 0.0
Best treated lambda: 221.221629107045
Best treated CV AUC: 0.7195



Best control alpha: 0.1
Best control lambda: 117.21022975334806
Best control CV AUC: 0.6472



---
## OPTIONAL TEST AUC FOR CPU-ONLY GLMNET COMPARISON
---


In [24]:
if enet_treated is not None and enet_control is not None:
    pred_treated_cv_glmnet = enet_treated['best_model'].predict_proba(x_test.iloc[test_treated_pos])[:, 1]
    auc_treated_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_glmnet, 'GLMNET Treated CV-tuned test')

    pred_control_cv_glmnet = enet_control['best_model'].predict_proba(x_test.iloc[test_control_pos])[:, 1]
    auc_control_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_glmnet, 'GLMNET Control CV-tuned test')
else:
    pred_treated_cv_glmnet = None
    pred_control_cv_glmnet = None
    auc_treated_cv_glmnet = np.nan
    auc_control_cv_glmnet = np.nan
    print('Skipped GLMNET test AUC because CPU-only GLMNET training was skipped.')
print()


GLMNET Treated CV-tuned test AUC: 0.9044
GLMNET Control CV-tuned test AUC: 0.6600



---
## 15. SCORE TEST SETS
---


In [25]:
p_treated_xgboost = model_treated.predict(make_dmatrix(x_test))
p_control_xgboost = model_control.predict(make_dmatrix(x_test))
results_test_xgboost = build_uplift_results(test_df, p_treated_xgboost, p_control_xgboost)

# Backward-compatible alias for the primary XGBoost result.
results_test = results_test_xgboost
print_highest_benefit(results_test_xgboost, 'XGBoost')

if enet_treated is not None and enet_control is not None:
    p_treated_glmnet = enet_treated['best_model'].predict_proba(x_test)[:, 1]
    p_control_glmnet = enet_control['best_model'].predict_proba(x_test)[:, 1]
    results_test_glmnet = build_uplift_results(test_df, p_treated_glmnet, p_control_glmnet)
    print_highest_benefit(results_test_glmnet, 'GLMNET')
else:
    p_treated_glmnet = None
    p_control_glmnet = None
    results_test_glmnet = None
    print('Skipped GLMNET test scoring because CPU-only GLMNET training was skipped.')
    print()


XGBoost top 20 highest-benefit members:


,outcome_ed_90d,intervention_flag,pred_ed_if_treated,pred_ed_if_control,benefit_score,uplift_decile
79,0.0,0.0,0.119394,0.179775,0.060380,1
640,0.0,0.0,0.122150,0.182374,0.060224,1
604,0.0,0.0,0.123718,0.183734,0.060016,1
18,0.0,0.0,0.113752,0.173747,0.059995,1
569,1.0,0.0,0.120756,0.179350,0.058594,1
139,0.0,0.0,0.115449,0.170614,0.055165,1
516,0.0,0.0,0.111028,0.166097,0.055069,1
206,0.0,1.0,0.111028,0.165699,0.054670,1
937,0.0,0.0,0.125883,0.178097,0.052214,1
561,0.0,1.0,0.122932,0.174545,0.051613,1



GLMNET top 20 highest-benefit members:


,outcome_ed_90d,intervention_flag,pred_ed_if_treated,pred_ed_if_control,benefit_score,uplift_decile
600,0.0,0.0,0.035484,0.087115,0.051631,1
748,0.0,0.0,0.032745,0.081029,0.048285,1
349,0.0,0.0,0.039085,0.086845,0.047760,1
39,0.0,0.0,0.035338,0.081786,0.046448,1
206,0.0,1.0,0.038330,0.084466,0.046136,1
139,0.0,0.0,0.045699,0.091342,0.045643,1
315,0.0,1.0,0.034325,0.079271,0.044945,1
426,0.0,1.0,0.035709,0.080292,0.044583,1
250,0.0,1.0,0.040336,0.084510,0.044174,1
851,0.0,0.0,0.036448,0.080521,0.044073,1


---
## 16. DECILE SUMMARIES
---


In [26]:
decile_summary_xgboost = summarize_uplift_deciles(results_test_xgboost)

# Backward-compatible alias for the primary XGBoost summary.
decile_summary = decile_summary_xgboost

print('XGBoost decile summary:')
display(decile_summary_xgboost)
print()

if results_test_glmnet is not None:
    decile_summary_glmnet = summarize_uplift_deciles(results_test_glmnet)
    print('GLMNET decile summary:')
    display(decile_summary_glmnet)
    print()
else:
    decile_summary_glmnet = None


XGBoost decile summary:


,uplift_decile,n,avg_benefit_score,observed_ed_rate,treated_pct,avg_pred_ed_if_treated,avg_pred_ed_if_control
0,1,30,0.048251,0.100000,0.300000,0.120074,0.168325
1,2,30,0.027164,0.133333,0.533333,0.119016,0.146180
2,3,30,0.008736,0.000000,0.400000,0.114181,0.122917
3,4,30,-0.002640,0.033333,0.266667,0.113461,0.110822
4,5,30,-0.007040,0.100000,0.333333,0.112870,0.105830
5,6,30,-0.010456,0.066667,0.533333,0.112809,0.102353
6,7,30,-0.014240,0.066667,0.466667,0.114410,0.100170
7,8,30,-0.018811,0.000000,0.333333,0.113318,0.094506
8,9,30,-0.024433,0.000000,0.300000,0.118714,0.094281
9,10,30,-0.033480,0.100000,0.466667,0.125043,0.091563



GLMNET decile summary:


,uplift_decile,n,avg_benefit_score,observed_ed_rate,treated_pct,avg_pred_ed_if_treated,avg_pred_ed_if_control
0,1,30,0.044022,0.066667,0.366667,0.035710,0.079732
1,2,30,0.040271,0.000000,0.433333,0.034523,0.074794
2,3,30,0.038177,0.000000,0.366667,0.033556,0.071733
3,4,30,0.036518,0.033333,0.366667,0.033323,0.069842
4,5,30,0.035405,0.066667,0.400000,0.036684,0.072090
5,6,30,0.034239,0.033333,0.333333,0.037637,0.071876
6,7,30,0.032882,0.066667,0.566667,0.037847,0.070729
7,8,30,0.031156,0.133333,0.400000,0.041132,0.072288
8,9,30,0.028667,0.000000,0.333333,0.044195,0.072862
9,10,30,0.023024,0.200000,0.366667,0.058430,0.081454


---
## X-LEARNER TREATMENT EFFECT COMPARISON
---

This section trains X-learner treatment-effect models for both XGBoost and GLMNet. The goal is not to replace the T-learner output, but to provide a second uplift framework that can be compared against the T-learner benefit ranking for consistency.

### How the X-learner works in this PRISM uplift workflow

```mermaid
flowchart TD
    A["PRISM training data<br/>Member predictors X, intervention flag W, ED outcome Y"] --> B{"Split by observed intervention status"}

    B --> C["Treated members<br/>W = 1"]
    B --> D["Control members<br/>W = 0"]

    C --> E["Train treated outcome model<br/>mu1(X) = predicted ED risk if treated"]
    D --> F["Train control outcome model<br/>mu0(X) = predicted ED risk if not treated"]

    E --> G["Predict treated risk for control members<br/>mu1(X_control)"]
    F --> H["Predict control risk for treated members<br/>mu0(X_treated)"]

    H --> I["Impute treated-group benefit<br/>D1 = mu0(X_treated) - observed Y_treated"]
    G --> J["Impute control-group benefit<br/>D0 = observed Y_control - mu1(X_control)"]

    I --> K["Train effect model on treated members<br/>tau1(X) learns benefit pattern for W = 1"]
    J --> L["Train effect model on control members<br/>tau0(X) learns benefit pattern for W = 0"]

    A --> M["Train propensity model<br/>e(X) = probability member received intervention"]

    K --> N["Score test members<br/>tau1(X_test)"]
    L --> O["Score test members<br/>tau0(X_test)"]
    M --> P["Score test members<br/>e(X_test), clipped to 0.05-0.95"]

    N --> Q["Weighted X-learner benefit score<br/>benefit = e(X) * tau0(X) + (1 - e(X)) * tau1(X)"]
    O --> Q
    P --> Q

    Q --> R["Rank members into uplift deciles<br/>Decile 1 = highest predicted treatment benefit"]
    R --> S["Compare X-learner ranking with T-learner ranking<br/>correlations, top-decile overlap, decile summaries"]
```

In this notebook, a larger X-learner `benefit_score` means the intervention is predicted to reduce 90-day ED risk more for that member. The X-learner differs from the T-learner by first creating imputed individual benefit labels for treated and control members, then learning those benefit patterns directly and blending them with propensity weights.

---


In [27]:
# X-learner implementation for treatment-effect consistency checks.
# Uses the already-trained T-learner outcome models to impute individual treatment effects,
# then trains second-stage treatment-effect models on those imputed effects.

xlearner_root_folder = ensure_output_folder(output_folder / 'X-Learner')
xlearner_xgboost_output_folder = ensure_output_folder(xlearner_root_folder / 'XGBoost')
xlearner_glmnet_output_folder = ensure_output_folder(xlearner_root_folder / 'GLMNet')


def fit_xgb_regression_model(x_matrix, y, model_label, seed=123):
    params = xgb_training_params(
        XGB_GPU_PARAMS,
        {
            'max_depth': 3,
            'eta': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 1,
            'objective': 'reg:squarederror',
        },
        eval_metric='rmse',
        seed=seed,
    )
    model = xgb.train(
        params=params,
        dtrain=make_dmatrix(x_matrix, np.asarray(y, dtype=float)),
        num_boost_round=200,
        verbose_eval=False,
    )
    assert_xgb_booster_uses_cuda(model, model_label)
    return model


class PrefitScaledRegressionPipeline:
    def __init__(self, scaler, model):
        self.named_steps = {
            'standardscaler': scaler,
            'elasticnetcv': model,
        }

    def predict(self, x_matrix):
        return self.named_steps['elasticnetcv'].predict(
            self.named_steps['standardscaler'].transform(x_matrix)
        )


def fit_glmnet_regression_model(x_matrix, y, seed=123, prefit_scaler=None):
    y_array = np.asarray(y, dtype=float)
    folds = int(min(5, len(y_array)))
    if folds < 2:
        raise ValueError('Need at least two rows for GLMNET X-learner regression.')
    reg_model = ElasticNetCV(
        l1_ratio=np.round(np.arange(0.0, 1.01, 0.1), 1),
        alphas=np.logspace(-4, 2, 50),
        cv=folds,
        max_iter=10000,
        random_state=seed,
    )
    if prefit_scaler is None:
        pipeline = make_pipeline(StandardScaler(), reg_model)
        pipeline.fit(x_matrix, y_array)
    else:
        x_scaled = prefit_scaler.transform(x_matrix)
        reg_model.fit(x_scaled, y_array)
        pipeline = PrefitScaledRegressionPipeline(prefit_scaler, reg_model)
    return pipeline


def fit_propensity_model(x_matrix, treatment, seed=123):
    treatment_array = np.asarray(treatment, dtype=float)
    class_counts = pd.Series(treatment_array).value_counts()
    folds = int(min(5, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two treatment classes with at least two rows each for propensity modeling.')
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    propensity_model = LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 30),
        cv=cv,
        penalty='elasticnet',
        solver='saga',
        l1_ratios=[0.5],
        scoring='roc_auc',
        max_iter=10000,
        random_state=seed,
        refit=True,
    )
    pipeline = make_pipeline(StandardScaler(), propensity_model)
    pipeline.fit(x_matrix, treatment_array)
    return pipeline


def clipped_propensity(propensity_model, x_matrix, lower=0.05, upper=0.95):
    propensity = propensity_model.predict_proba(x_matrix)[:, 1]
    return np.clip(propensity, lower, upper)


def build_xlearner_results(base_df, pred_treated, pred_control, cate_score, propensity_score, learner_label):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['t_learner_benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['benefit_score'] = cate_score
    results['xlearner_propensity_score'] = propensity_score
    results['uplift_bad_outcome'] = -results['benefit_score']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    results['learner_framework'] = 'X-Learner'
    results['model'] = learner_label
    return results


def xlearner_consistency_summary(t_results, x_results, model_label):
    merged = pd.DataFrame(
        {
            't_learner_benefit_score': t_results['benefit_score'].to_numpy(),
            'x_learner_benefit_score': x_results['benefit_score'].to_numpy(),
            't_learner_decile': t_results['uplift_decile'].to_numpy(),
            'x_learner_decile': x_results['uplift_decile'].to_numpy(),
        },
        index=t_results.index,
    )
    t_top = set(merged.index[merged['t_learner_decile'] == 1])
    x_top = set(merged.index[merged['x_learner_decile'] == 1])
    top_overlap = len(t_top & x_top) / len(t_top) if t_top else np.nan
    return pd.DataFrame(
        [
            {
                'model': model_label,
                'pearson_benefit_score_corr': merged['t_learner_benefit_score'].corr(merged['x_learner_benefit_score'], method='pearson'),
                'spearman_benefit_score_corr': merged['t_learner_benefit_score'].corr(merged['x_learner_benefit_score'], method='spearman'),
                'top_decile_overlap_pct': top_overlap,
                't_learner_mean_benefit_score': merged['t_learner_benefit_score'].mean(),
                'x_learner_mean_benefit_score': merged['x_learner_benefit_score'].mean(),
            }
        ]
    )


x_train_all = pd.concat([x_treated, x_control], axis=0)
t_train_all = pd.concat(
    [
        pd.Series(np.ones(len(x_treated)), index=x_treated.index),
        pd.Series(np.zeros(len(x_control)), index=x_control.index),
    ],
    axis=0,
)
propensity_model = fit_propensity_model(x_train_all, t_train_all)
propensity_train_all = clipped_propensity(propensity_model, x_train_all)
propensity_test = clipped_propensity(propensity_model, x_test)

shared_propensity_scores = pd.concat(
    [
        pd.DataFrame({
            'member_id': pd.concat([train_treated['member_id'], train_control['member_id']], axis=0).to_numpy(),
            'split': 'train',
            'propensity_score': propensity_train_all,
            'propensity_model': 'GLMNet elastic-net logistic',
            'seed': 123,
        }),
        pd.DataFrame({
            'member_id': test_df['member_id'].to_numpy(),
            'split': 'test',
            'propensity_score': propensity_test,
            'propensity_model': 'GLMNet elastic-net logistic',
            'seed': 123,
        }),
    ],
    ignore_index=True,
).sort_values('member_id').reset_index(drop=True)
shared_propensity_path = xlearner_root_folder / 'shared_propensity_scores.csv'
shared_propensity_scores.to_csv(shared_propensity_path, index=False)
print('Shared member-level propensity scores written to:', shared_propensity_path)

# XGBoost X-learner: impute benefit effects on the same scale as the T-learner.
# Positive benefit means lower ED risk under treatment: control outcome/risk - treated outcome/risk.
xgb_mu0_on_treated = model_control.predict(make_dmatrix(x_treated))
xgb_mu1_on_control = model_treated.predict(make_dmatrix(x_control))
xgb_effect_treated = xgb_mu0_on_treated - np.asarray(y_treated, dtype=float)
xgb_effect_control = np.asarray(y_control, dtype=float) - xgb_mu1_on_control

xgb_tau_treated_model = fit_xgb_regression_model(
    x_treated,
    xgb_effect_treated,
    'XGBoost X-learner treated-effect model',
)
xgb_tau_control_model = fit_xgb_regression_model(
    x_control,
    xgb_effect_control,
    'XGBoost X-learner control-effect model',
)

xgb_tau_treated_test = xgb_tau_treated_model.predict(make_dmatrix(x_test))
xgb_tau_control_test = xgb_tau_control_model.predict(make_dmatrix(x_test))
xgb_xlearner_benefit = propensity_test * xgb_tau_control_test + (1 - propensity_test) * xgb_tau_treated_test

results_test_xlearner_xgboost = build_xlearner_results(
    test_df,
    p_treated_xgboost,
    p_control_xgboost,
    xgb_xlearner_benefit,
    propensity_test,
    'XGBoost',
)
decile_summary_xlearner_xgboost = summarize_uplift_deciles(results_test_xlearner_xgboost)
print('XGBoost X-learner decile summary:')
display(decile_summary_xlearner_xgboost)
print_highest_benefit(results_test_xlearner_xgboost, 'XGBoost X-Learner')

results_test_xlearner_xgboost.to_csv(xlearner_xgboost_output_folder / 'xlearner_scored_test_output.csv', index=False)
decile_summary_xlearner_xgboost.to_csv(xlearner_xgboost_output_folder / 'xlearner_decile_summary.csv', index=False)

# GLMNET X-learner: impute benefit effects on the same control-minus-treated scale,
# then fit elastic-net second-stage treatment-effect regressions.
if enet_treated is not None and enet_control is not None:
    glmnet_mu0_on_treated = enet_control['best_model'].predict_proba(x_treated)[:, 1]
    glmnet_mu1_on_control = enet_treated['best_model'].predict_proba(x_control)[:, 1]
    glmnet_effect_treated = glmnet_mu0_on_treated - np.asarray(y_treated, dtype=float)
    glmnet_effect_control = np.asarray(y_control, dtype=float) - glmnet_mu1_on_control

    xlearner_effect_shared_scaler = StandardScaler().fit(pd.concat([x_treated, x_control], axis=0))
    print('GLMNET X-learner shared effect scaler fit on combined treated/control training matrix.')
    glmnet_tau_treated_model = fit_glmnet_regression_model(
        x_treated,
        glmnet_effect_treated,
        prefit_scaler=xlearner_effect_shared_scaler,
    )
    glmnet_tau_control_model = fit_glmnet_regression_model(
        x_control,
        glmnet_effect_control,
        prefit_scaler=xlearner_effect_shared_scaler,
    )

    glmnet_tau_treated_test = glmnet_tau_treated_model.predict(x_test)
    glmnet_tau_control_test = glmnet_tau_control_model.predict(x_test)
    glmnet_xlearner_benefit = propensity_test * glmnet_tau_control_test + (1 - propensity_test) * glmnet_tau_treated_test

    results_test_xlearner_glmnet = build_xlearner_results(
        test_df,
        p_treated_glmnet,
        p_control_glmnet,
        glmnet_xlearner_benefit,
        propensity_test,
        'GLMNET',
    )
    decile_summary_xlearner_glmnet = summarize_uplift_deciles(results_test_xlearner_glmnet)
    print('GLMNET X-learner decile summary:')
    display(decile_summary_xlearner_glmnet)
    print_highest_benefit(results_test_xlearner_glmnet, 'GLMNET X-Learner')

    results_test_xlearner_glmnet.to_csv(xlearner_glmnet_output_folder / 'xlearner_scored_test_output.csv', index=False)
    decile_summary_xlearner_glmnet.to_csv(xlearner_glmnet_output_folder / 'xlearner_decile_summary.csv', index=False)
else:
    xlearner_effect_shared_scaler = None
    glmnet_tau_treated_model = None
    glmnet_tau_control_model = None
    results_test_xlearner_glmnet = None
    decile_summary_xlearner_glmnet = None
    print('Skipped GLMNET X-learner because GLMNET T-learner outcome models were not available.')

consistency_frames = [
    xlearner_consistency_summary(results_test_xgboost, results_test_xlearner_xgboost, 'XGBoost'),
]
if results_test_glmnet is not None and results_test_xlearner_glmnet is not None:
    consistency_frames.append(
        xlearner_consistency_summary(results_test_glmnet, results_test_xlearner_glmnet, 'GLMNET')
    )

xlearner_consistency = pd.concat(consistency_frames, ignore_index=True)
xlearner_consistency.to_csv(xlearner_root_folder / 'xlearner_vs_tlearner_consistency_summary.csv', index=False)
print('X-learner versus T-learner consistency summary:')
display(xlearner_consistency)
print('X-learner outputs saved to:', xlearner_root_folder)


def save_paired_glmnet_benefit_decile_charts(t_decile_df, x_decile_df):
    max_benefit = max(
        t_decile_df['avg_benefit_score'].max(),
        x_decile_df['avg_benefit_score'].max(),
    )
    shared_y_max = np.ceil((max_benefit * 1.08) / 0.005) * 0.005

    chart_specs = [
        (
            t_decile_df,
            glmnet_output_folder / 'dashboard_avg_benefit_by_decile.png',
            'GLMNet T-Learner: Average Predicted Benefit by Decile',
        ),
        (
            x_decile_df,
            xlearner_glmnet_output_folder / 'dashboard_avg_benefit_by_decile.png',
            'GLMNet X-Learner: Average Predicted Benefit by Decile',
        ),
    ]

    for decile_df, chart_path, title in chart_specs:
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        ax.bar(
            decile_df['uplift_decile'].astype(str),
            decile_df['avg_benefit_score'],
            color='#4f76b5',
        )
        ax.set_title(title)
        ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
        ax.set_ylabel('Average Predicted Benefit')
        ax.set_ylim(0, shared_y_max)
        ax.grid(axis='y', alpha=0.35)
        ax.set_axisbelow(True)
        fig.tight_layout()
        fig.savefig(chart_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved paired GLMNet benefit decile charts with shared y-axis max {shared_y_max:.3f}.')


if decile_summary_glmnet is not None and decile_summary_xlearner_glmnet is not None:
    save_paired_glmnet_benefit_decile_charts(
        decile_summary_glmnet,
        decile_summary_xlearner_glmnet,
    )
else:
    print('Skipped paired GLMNet benefit decile charts because one or more decile summaries were unavailable.')


Shared member-level propensity scores written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/shared_propensity_scores.csv


XGBoost X-learner treated-effect model booster config confirms CUDA/GPU training.


XGBoost X-learner control-effect model booster config confirms CUDA/GPU training.
XGBoost X-learner decile summary:


,uplift_decile,n,avg_benefit_score,observed_ed_rate,treated_pct,avg_pred_ed_if_treated,avg_pred_ed_if_control
0,1,30,0.150855,0.133333,0.533333,0.121210,0.142748
1,2,30,0.093749,0.033333,0.333333,0.116613,0.134848
2,3,30,0.068619,0.066667,0.366667,0.113321,0.117097
3,4,30,0.051437,0.033333,0.366667,0.113421,0.109386
4,5,30,0.040168,0.033333,0.333333,0.113035,0.109067
5,6,30,0.025908,0.066667,0.200000,0.113273,0.107512
6,7,30,0.010971,0.033333,0.366667,0.115008,0.101500
7,8,30,-0.004885,0.066667,0.400000,0.117246,0.104114
8,9,30,-0.027671,0.033333,0.600000,0.118125,0.101308
9,10,30,-0.098908,0.100000,0.433333,0.122644,0.109368


XGBoost X-Learner top 20 highest-benefit members:


...

,outcome_ed_90d,intervention_flag,pred_ed_if_treated,pred_ed_if_control,benefit_score,uplift_decile
444,0.0,1.0,0.085769,0.104078,0.124568,1
469,0.0,0.0,0.096727,0.108375,0.120883,1
672,1.0,0.0,0.050514,0.084566,0.094444,1
606,1.0,0.0,0.069345,0.094118,0.093983,1
474,0.0,1.0,0.062590,0.086867,0.079824,1
44,0.0,0.0,0.057027,0.084489,0.078521,1
782,0.0,1.0,0.055624,0.083181,0.076942,1
871,0.0,1.0,0.051476,0.086828,0.073815,1
431,0.0,1.0,0.057492,0.099190,0.071049,1
516,0.0,0.0,0.053883,0.086794,0.070356,1



X-learner versus T-learner consistency summary:


,model,pearson_benefit_score_corr,spearman_benefit_score_corr,top_decile_overlap_pct,t_learner_mean_benefit_score,x_learner_mean_benefit_score
0,XGBoost,0.470129,0.514313,0.300000,-0.002695,0.031024
1,GLMNET,-0.076110,0.160788,0.166667,0.034436,0.037714


X-learner outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner


Saved paired GLMNet benefit decile charts with shared y-axis max 0.080.


---
## WRITE-UP PROBABILITY CALIBRATION AND BRIER SCORES
---


In [28]:
def brier_scores_for_results(results, model_label):
    if results is None:
        return None

    rows = []
    for group_label, treatment_value, pred_col in [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        if subgroup.empty:
            rows.append({
                'model': model_label,
                'group': group_label,
                'n': 0,
                'observed_ed_rate': np.nan,
                'avg_predicted_ed_rate': np.nan,
                'brier_score': np.nan,
            })
            continue

        rows.append({
            'model': model_label,
            'group': group_label,
            'n': len(subgroup),
            'observed_ed_rate': subgroup['outcome_ed_90d'].mean(),
            'avg_predicted_ed_rate': subgroup[pred_col].mean(),
            'brier_score': brier_score_loss(subgroup['outcome_ed_90d'], subgroup[pred_col]),
        })

    return pd.DataFrame(rows)


def calibration_tables_for_results(results, model_label):
    if results is None:
        return None, None

    calibration_parts = []
    for group_label, treatment_value, pred_col in [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        if subgroup.empty:
            continue

        subgroup['pred_risk_decile'] = ntile_desc(subgroup[pred_col], 10).to_numpy()
        by_decile = (
            subgroup
            .groupby('pred_risk_decile', as_index=False)
            .agg(
                n=('outcome_ed_90d', 'size'),
                avg_predicted_ed_rate=(pred_col, 'mean'),
                observed_ed_rate=('outcome_ed_90d', 'mean'),
            )
            .sort_values('pred_risk_decile')
        )
        by_decile.insert(0, 'group', group_label)
        by_decile.insert(0, 'model', model_label)
        by_decile['calibration_error'] = by_decile['observed_ed_rate'] - by_decile['avg_predicted_ed_rate']
        by_decile['abs_calibration_error'] = by_decile['calibration_error'].abs()
        calibration_parts.append(by_decile)

    if not calibration_parts:
        return None, None

    calibration_by_decile = pd.concat(calibration_parts, ignore_index=True)
    calibration_summary_rows = []
    for (model_name, group_label), frame in calibration_by_decile.groupby(['model', 'group']):
        calibration_summary_rows.append(
            {
                'model': model_name,
                'group': group_label,
                'n': frame['n'].sum(),
                'mean_abs_calibration_error': frame['abs_calibration_error'].mean(),
                'weighted_mean_abs_calibration_error': np.average(
                    frame['abs_calibration_error'],
                    weights=frame['n'],
                ),
                'max_abs_calibration_error': frame['abs_calibration_error'].max(),
            }
        )
    calibration_summary = pd.DataFrame(calibration_summary_rows)
    return calibration_by_decile, calibration_summary


def save_calibration_plot(calibration_by_decile, folder, model_label):
    if calibration_by_decile is None or calibration_by_decile.empty:
        return

    groups = [group for group in ['Control', 'Treated'] if group in set(calibration_by_decile['group'])]
    fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 5), sharey=True)
    if len(groups) == 1:
        axes = [axes]

    max_rate = max(
        calibration_by_decile['avg_predicted_ed_rate'].max(),
        calibration_by_decile['observed_ed_rate'].max(),
        0.01,
    )

    for ax, group_label in zip(axes, groups):
        group_df = calibration_by_decile[calibration_by_decile['group'] == group_label].sort_values('pred_risk_decile')
        x_values = np.arange(len(group_df))
        bar_width = 0.38

        ax.bar(
            x_values - bar_width / 2,
            group_df['avg_predicted_ed_rate'],
            width=bar_width,
            label='Avg predicted ED rate',
            color='#4C78A8',
        )
        ax.bar(
            x_values + bar_width / 2,
            group_df['observed_ed_rate'],
            width=bar_width,
            label='Observed ED rate',
            color='#F58518',
        )

        for x_pos, n_value in zip(x_values, group_df['n']):
            ax.text(x_pos, max_rate * 1.03, f'n={int(n_value)}', ha='center', va='bottom', fontsize=8, rotation=90)

        ax.set_title(f'{model_label}: {group_label} Calibration')
        ax.set_xlabel('Predicted Risk Decile: 1 = Highest Predicted Risk')
        ax.set_xticks(x_values)
        ax.set_xticklabels(group_df['pred_risk_decile'].astype(str))
        ax.set_ylim(0, max_rate * 1.22)
        ax.grid(axis='y', alpha=0.25)

    axes[0].set_ylabel('ED Rate')
    axes[-1].legend(loc='upper right')
    fig.suptitle(f'{model_label}: Predicted vs Observed ED Rate by Risk Decile', y=1.03)
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_calibration_plot.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def save_probability_evaluation_outputs(results, folder, model_label):
    brier_df = brier_scores_for_results(results, model_label)
    calibration_by_decile, calibration_summary = calibration_tables_for_results(results, model_label)

    if brier_df is not None:
        brier_df.to_csv(folder / 'model_brier_scores.csv', index=False)
        print(f'{model_label} Brier scores:')
        display(brier_df)

    if calibration_by_decile is not None:
        calibration_by_decile.to_csv(folder / 'calibration_by_decile.csv', index=False)
        calibration_summary.to_csv(folder / 'calibration_summary.csv', index=False)
        save_calibration_plot(calibration_by_decile, folder, model_label)
        print(f'{model_label} calibration summary:')
        display(calibration_summary)
        print()

    return brier_df, calibration_by_decile, calibration_summary


brier_xgboost, calibration_by_decile_xgboost, calibration_summary_xgboost = save_probability_evaluation_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None:
    brier_glmnet, calibration_by_decile_glmnet, calibration_summary_glmnet = save_probability_evaluation_outputs(
        results_test_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    brier_glmnet = None
    calibration_by_decile_glmnet = None
    calibration_summary_glmnet = None
    print('Skipped GLMNET Brier/calibration outputs because GLMNET scoring was not available.')


XGBoost Brier scores:


,model,group,n,observed_ed_rate,avg_predicted_ed_rate,brier_score
0,XGBoost,Treated,118,0.042373,0.116990,0.045232
1,XGBoost,Control,182,0.071429,0.113756,0.068098


XGBoost calibration summary:


,model,group,n,mean_abs_calibration_error,weighted_mean_abs_calibration_error,max_abs_calibration_error
0,XGBoost,Control,182,0.064428,0.064377,0.100418
1,XGBoost,Treated,118,0.080065,0.080303,0.118597



GLMNET Brier scores:


,model,group,n,observed_ed_rate,avg_predicted_ed_rate,brier_score
0,GLMNET,Treated,118,0.042373,0.040296,0.039236
1,GLMNET,Control,182,0.071429,0.073231,0.065799


GLMNET calibration summary:


,model,group,n,mean_abs_calibration_error,weighted_mean_abs_calibration_error,max_abs_calibration_error
0,GLMNET,Control,182,0.050915,0.051123,0.121632
1,GLMNET,Treated,118,0.061030,0.061494,0.269505


---
## WRITE-UP FACTUAL OUTCOME MODEL DIAGNOSTICS
---


In [29]:
def factual_event_count_summary(train_df, test_df):
    rows = []
    for split_name, frame in [('Train', train_df), ('Test', test_df)]:
        for group_label, treatment_value in [('Treated', 1), ('Control', 0)]:
            subgroup = frame[frame['intervention_flag'] == treatment_value]
            n = len(subgroup)
            events = int(subgroup['outcome_ed_90d'].sum()) if n else 0
            rows.append(
                {
                    'split': split_name,
                    'group': group_label,
                    'n': n,
                    'positive_ed_events': events,
                    'negative_ed_events': n - events,
                    'event_rate': events / n if n else np.nan,
                }
            )
    return pd.DataFrame(rows)


def factual_prediction_diagnostics(results, model_label):
    separation_rows = []
    threshold_rows = []
    range_rows = []
    specs = [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]

    for group_label, treatment_value, pred_col in specs:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        n = len(subgroup)
        events = int(subgroup['outcome_ed_90d'].sum()) if n else 0
        non_events = n - events
        positives = subgroup[subgroup['outcome_ed_90d'] == 1]
        negatives = subgroup[subgroup['outcome_ed_90d'] == 0]
        auc_value = roc_auc_score(subgroup['outcome_ed_90d'], subgroup[pred_col]) if subgroup['outcome_ed_90d'].nunique() == 2 else np.nan

        separation_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'n': n,
                'positive_ed_events': events,
                'negative_ed_events': non_events,
                'event_rate': events / n if n else np.nan,
                'auc': auc_value,
                'avg_pred_actual_positive': positives[pred_col].mean() if len(positives) else np.nan,
                'avg_pred_actual_negative': negatives[pred_col].mean() if len(negatives) else np.nan,
                'avg_pred_positive_minus_negative': (
                    positives[pred_col].mean() - negatives[pred_col].mean()
                    if len(positives) and len(negatives)
                    else np.nan
                ),
                'median_pred_actual_positive': positives[pred_col].median() if len(positives) else np.nan,
                'median_pred_actual_negative': negatives[pred_col].median() if len(negatives) else np.nan,
                'median_pred_positive_minus_negative': (
                    positives[pred_col].median() - negatives[pred_col].median()
                    if len(positives) and len(negatives)
                    else np.nan
                ),
            }
        )

        threshold = events / n if n else np.nan
        predicted_positive = subgroup[pred_col] >= threshold if n and not pd.isna(threshold) else pd.Series(False, index=subgroup.index)
        actual_positive = subgroup['outcome_ed_90d'] == 1
        true_positive = int((predicted_positive & actual_positive).sum())
        false_positive = int((predicted_positive & ~actual_positive).sum())
        true_negative = int((~predicted_positive & ~actual_positive).sum())
        false_negative = int((~predicted_positive & actual_positive).sum())
        predicted_positive_n = true_positive + false_positive
        precision = true_positive / predicted_positive_n if predicted_positive_n else np.nan
        recall = true_positive / events if events else np.nan

        threshold_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'threshold_rule': 'factual_group_test_event_rate',
                'threshold': threshold,
                'n': n,
                'actual_positive_n': events,
                'actual_negative_n': non_events,
                'predicted_positive_n': predicted_positive_n,
                'predicted_negative_n': true_negative + false_negative,
                'true_positive': true_positive,
                'false_positive': false_positive,
                'true_negative': true_negative,
                'false_negative': false_negative,
                'precision': precision,
                'recall': recall,
            }
        )

        predictions = subgroup[pred_col].dropna()
        range_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'n': n,
                'min_pred': predictions.min() if len(predictions) else np.nan,
                'p10_pred': predictions.quantile(0.10) if len(predictions) else np.nan,
                'median_pred': predictions.median() if len(predictions) else np.nan,
                'mean_pred': predictions.mean() if len(predictions) else np.nan,
                'p90_pred': predictions.quantile(0.90) if len(predictions) else np.nan,
                'max_pred': predictions.max() if len(predictions) else np.nan,
            }
        )

    return pd.DataFrame(separation_rows), pd.DataFrame(threshold_rows), pd.DataFrame(range_rows)


event_count_summary = factual_event_count_summary(train_df, test_df)
event_count_summary_path = output_folder / 'factual_event_count_summary.csv'
event_count_summary.to_csv(event_count_summary_path, index=False)

diagnostic_frames = []
threshold_frames = []
range_frames = []

sep_xgboost, threshold_xgboost, range_xgboost = factual_prediction_diagnostics(results_test_xgboost, 'XGBoost')
diagnostic_frames.append(sep_xgboost)
threshold_frames.append(threshold_xgboost)
range_frames.append(range_xgboost)

if results_test_glmnet is not None:
    sep_glmnet, threshold_glmnet, range_glmnet = factual_prediction_diagnostics(results_test_glmnet, 'GLMNET')
    diagnostic_frames.append(sep_glmnet)
    threshold_frames.append(threshold_glmnet)
    range_frames.append(range_glmnet)

factual_prediction_separation = pd.concat(diagnostic_frames, ignore_index=True)
factual_event_rate_threshold_classification = pd.concat(threshold_frames, ignore_index=True)
factual_prediction_ranges = pd.concat(range_frames, ignore_index=True)

factual_prediction_separation_path = output_folder / 'factual_prediction_separation.csv'
factual_prediction_ranges_path = output_folder / 'factual_prediction_ranges.csv'

factual_prediction_separation.to_csv(factual_prediction_separation_path, index=False)
factual_prediction_ranges.to_csv(factual_prediction_ranges_path, index=False)

print('Factual train/test event counts:')
display(event_count_summary)
print('Factual positive-vs-negative prediction separation:')
display(factual_prediction_separation)
print('Event-rate threshold classification:')
display(factual_event_rate_threshold_classification)
print('Prediction ranges:')
display(factual_prediction_ranges)
print('Factual diagnostic outputs written to:', output_folder)


Factual train/test event counts:


,split,group,n,positive_ed_events,negative_ed_events,event_rate
0,Train,Treated,276,11,265,0.039855
1,Train,Control,424,31,393,0.073113
2,Test,Treated,118,5,113,0.042373
3,Test,Control,182,13,169,0.071429


Factual positive-vs-negative prediction separation:


,model,group,prediction_column,n,positive_ed_events,negative_ed_events,event_rate,auc,avg_pred_actual_positive,avg_pred_actual_negative,avg_pred_positive_minus_negative,median_pred_actual_positive,median_pred_actual_negative,median_pred_positive_minus_negative
0,XGBoost,Treated,pred_ed_if_treated,118,5,113,0.042373,0.835398,0.128859,0.116465,0.012394,0.128578,0.111028,0.017550
1,XGBoost,Control,pred_ed_if_control,182,13,169,0.071429,0.583295,0.118378,0.113400,0.004978,0.111870,0.105486,0.006384
2,GLMNET,Treated,pred_ed_if_treated,118,5,113,0.042373,0.904425,0.057416,0.039539,0.017878,0.060629,0.036345,0.024284
3,GLMNET,Control,pred_ed_if_control,182,13,169,0.071429,0.659991,0.077374,0.072913,0.004462,0.074199,0.071328,0.002871


Event-rate threshold classification:


,model,group,prediction_column,threshold_rule,threshold,n,actual_positive_n,actual_negative_n,predicted_positive_n,predicted_negative_n,true_positive,false_positive,true_negative,false_negative,precision,recall
0,XGBoost,Treated,pred_ed_if_treated,factual_group_test_event_rate,0.042373,118,5,113,118,0,5,113,0,0,0.042373,1.000000
1,XGBoost,Control,pred_ed_if_control,factual_group_test_event_rate,0.071429,182,13,169,182,0,13,169,0,0,0.071429,1.000000
2,GLMNET,Treated,pred_ed_if_treated,factual_group_test_event_rate,0.042373,118,5,113,32,86,4,28,85,1,0.125000,0.800000
3,GLMNET,Control,pred_ed_if_control,factual_group_test_event_rate,0.071429,182,13,169,94,88,10,84,85,3,0.106383,0.769231


Prediction ranges:


,model,group,prediction_column,n,min_pred,p10_pred,median_pred,mean_pred,p90_pred,max_pred
0,XGBoost,Treated,pred_ed_if_treated,118,0.109634,0.109634,0.112069,0.116990,0.128756,0.152538
1,XGBoost,Control,pred_ed_if_control,182,0.081351,0.089460,0.105700,0.113756,0.155743,0.185438
2,GLMNET,Treated,pred_ed_if_treated,118,0.027427,0.031790,0.037154,0.040296,0.054499,0.085769
3,GLMNET,Control,pred_ed_if_control,182,0.060090,0.064379,0.071567,0.073231,0.084407,0.108375


Factual diagnostic outputs written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python


---
## WRITE-UP OBSERVED TREATED-CONTROL GAP BY UPLIFT DECILE
---


In [30]:
def wilson_score_interval(events, n, confidence=0.95):
    """Wilson score confidence interval for one binomial proportion."""
    if n == 0:
        return np.nan, np.nan

    z_lookup = {0.90: 1.644854, 0.95: 1.959964, 0.99: 2.575829}
    z_value = z_lookup.get(confidence, 1.959964)
    p_hat = events / n
    denominator = 1 + (z_value ** 2 / n)
    center = (p_hat + (z_value ** 2 / (2 * n))) / denominator
    half_width = (
        z_value
        * np.sqrt((p_hat * (1 - p_hat) / n) + (z_value ** 2 / (4 * n ** 2)))
        / denominator
    )
    return max(0, center - half_width), min(1, center + half_width)


def difference_in_proportions_ci(control_events, control_n, treated_events, treated_n, confidence=0.95):
    """Newcombe/Wilson CI for control observed ED rate minus treated observed ED rate."""
    if control_n == 0 or treated_n == 0:
        return np.nan, np.nan, np.nan

    control_rate = control_events / control_n
    treated_rate = treated_events / treated_n
    gap = control_rate - treated_rate
    control_lower, control_upper = wilson_score_interval(control_events, control_n, confidence)
    treated_lower, treated_upper = wilson_score_interval(treated_events, treated_n, confidence)
    gap_lower = control_lower - treated_upper
    gap_upper = control_upper - treated_lower
    return gap, gap_lower, gap_upper


def observed_gap_by_decile(results, model_label):
    if results is None:
        return None

    rows = []
    for decile, decile_df in results.groupby('uplift_decile'):
        treated = decile_df[decile_df['intervention_flag'] == 1]
        control = decile_df[decile_df['intervention_flag'] == 0]
        treated_events = int(treated['outcome_ed_90d'].sum()) if not treated.empty else 0
        control_events = int(control['outcome_ed_90d'].sum()) if not control.empty else 0
        treated_rate = treated_events / len(treated) if len(treated) else np.nan
        control_rate = control_events / len(control) if len(control) else np.nan
        gap, gap_ci_lower, gap_ci_upper = difference_in_proportions_ci(
            control_events,
            len(control),
            treated_events,
            len(treated),
        )
        rows.append({
            'model': model_label,
            'uplift_decile': decile,
            'n': len(decile_df),
            'treated_n': len(treated),
            'control_n': len(control),
            'treated_events': treated_events,
            'control_events': control_events,
            'observed_ed_rate': decile_df['outcome_ed_90d'].mean(),
            'treated_observed_ed_rate': treated_rate,
            'control_observed_ed_rate': control_rate,
            'observed_control_minus_treated_gap': gap,
            'observed_gap_ci_lower_95': gap_ci_lower,
            'observed_gap_ci_upper_95': gap_ci_upper,
            'avg_predicted_benefit': decile_df['benefit_score'].mean(),
            'predicted_benefit_within_observed_gap_ci_95': (
                bool(gap_ci_lower <= decile_df['benefit_score'].mean() <= gap_ci_upper)
                if pd.notna(gap_ci_lower) else np.nan
            ),
            'predicted_minus_observed_gap': decile_df['benefit_score'].mean() - gap if pd.notna(gap) else np.nan,
            'abs_predicted_minus_observed_gap': abs(decile_df['benefit_score'].mean() - gap) if pd.notna(gap) else np.nan,
            'avg_pred_ed_if_treated': decile_df['pred_ed_if_treated'].mean(),
            'avg_pred_ed_if_control': decile_df['pred_ed_if_control'].mean(),
            'treated_pct': decile_df['intervention_flag'].mean(),
            'ci_method': 'newcombe_wilson_difference_in_proportions',
        })

    return pd.DataFrame(rows).sort_values('uplift_decile').reset_index(drop=True)


def uplift_curve_by_decile(results, model_label):
    if results is None:
        return None

    rows = []
    sorted_deciles = sorted(results['uplift_decile'].dropna().unique())
    total_n = len(results)
    for max_decile in sorted_deciles:
        curve_df = results[results['uplift_decile'] <= max_decile].copy()
        treated = curve_df[curve_df['intervention_flag'] == 1]
        control = curve_df[curve_df['intervention_flag'] == 0]
        treated_events = int(treated['outcome_ed_90d'].sum()) if not treated.empty else 0
        control_events = int(control['outcome_ed_90d'].sum()) if not control.empty else 0
        gap, gap_ci_lower, gap_ci_upper = difference_in_proportions_ci(
            control_events,
            len(control),
            treated_events,
            len(treated),
        )
        rows.append({
            'model': model_label,
            'through_uplift_decile': max_decile,
            'population_fraction_targeted': len(curve_df) / total_n if total_n else np.nan,
            'n': len(curve_df),
            'treated_n': len(treated),
            'control_n': len(control),
            'treated_events': treated_events,
            'control_events': control_events,
            'avg_predicted_benefit': curve_df['benefit_score'].mean(),
            'observed_control_minus_treated_gap': gap,
            'observed_gap_ci_lower_95': gap_ci_lower,
            'observed_gap_ci_upper_95': gap_ci_upper,
            'predicted_benefit_within_observed_gap_ci_95': (
                bool(gap_ci_lower <= curve_df['benefit_score'].mean() <= gap_ci_upper)
                if pd.notna(gap_ci_lower) else np.nan
            ),
        })

    curve_df = pd.DataFrame(rows).sort_values('through_uplift_decile').reset_index(drop=True)
    return curve_df


def uplift_curve_summary(curve_df, model_label):
    if curve_df is None or curve_df.empty:
        return None

    valid_observed = curve_df[['population_fraction_targeted', 'observed_control_minus_treated_gap']].dropna()
    valid_predicted = curve_df[['population_fraction_targeted', 'avg_predicted_benefit']].dropna()
    observed_area = np.trapz(
        valid_observed['observed_control_minus_treated_gap'],
        valid_observed['population_fraction_targeted'],
    ) if len(valid_observed) >= 2 else np.nan
    predicted_area = np.trapz(
        valid_predicted['avg_predicted_benefit'],
        valid_predicted['population_fraction_targeted'],
    ) if len(valid_predicted) >= 2 else np.nan

    return pd.DataFrame([
        {
            'model': model_label,
            'observed_gap_area_under_curve': observed_area,
            'predicted_benefit_area_under_curve': predicted_area,
            'final_cumulative_observed_gap': curve_df['observed_control_minus_treated_gap'].iloc[-1],
            'final_cumulative_avg_predicted_benefit': curve_df['avg_predicted_benefit'].iloc[-1],
        }
    ])


def save_observed_gap_outputs(results, folder, model_label):
    gap_df = observed_gap_by_decile(results, model_label)
    if gap_df is None:
        return None, None, None

    gap_df.to_csv(folder / 'uplift_observed_gap_by_decile.csv', index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    x_values = np.arange(len(gap_df))
    gaps = gap_df['observed_control_minus_treated_gap'].to_numpy(dtype=float)
    ci_lower = gap_df['observed_gap_ci_lower_95'].to_numpy(dtype=float)
    ci_upper = gap_df['observed_gap_ci_upper_95'].to_numpy(dtype=float)
    yerr = np.vstack([gaps - ci_lower, ci_upper - gaps])
    ax.bar(x_values, gaps, color='#4c78a8')
    ax.errorbar(x_values, gaps, yerr=yerr, fmt='none', ecolor='#222222', capsize=4, linewidth=1)
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_xticks(x_values)
    ax.set_xticklabels(gap_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Observed Control-Treated ED Gap by Uplift Decile')
    ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
    ax.set_ylabel('Observed Control ED Rate - Treated ED Rate')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_observed_gap_by_decile.png', dpi=150)
    plt.close(fig)

    curve_df = uplift_curve_by_decile(results, model_label)
    curve_summary = uplift_curve_summary(curve_df, model_label)
    if curve_df is not None:
        curve_df.to_csv(folder / 'uplift_curve_by_decile.csv', index=False)

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(
            curve_df['population_fraction_targeted'],
            curve_df['avg_predicted_benefit'],
            marker='o',
            label='Average predicted benefit',
        )
        ax.plot(
            curve_df['population_fraction_targeted'],
            curve_df['observed_control_minus_treated_gap'],
            marker='o',
            label='Observed control-treated gap',
        )
        ax.fill_between(
            curve_df['population_fraction_targeted'].to_numpy(dtype=float),
            curve_df['observed_gap_ci_lower_95'].to_numpy(dtype=float),
            curve_df['observed_gap_ci_upper_95'].to_numpy(dtype=float),
            color='#f58518',
            alpha=0.18,
            label='Observed gap 95% CI',
        )
        ax.axhline(0, color='gray', linewidth=1)
        ax.set_title(f'{model_label}: Cumulative Uplift Curve by Targeted Fraction')
        ax.set_xlabel('Cumulative Fraction Targeted')
        ax.set_ylabel('ED Rate Difference / Predicted Benefit')
        ax.legend()
        fig.tight_layout()
        fig.savefig(folder / 'dashboard_uplift_curve_by_decile.png', dpi=150)
        plt.close(fig)

    if curve_summary is not None:
        curve_summary.to_csv(folder / 'uplift_curve_summary.csv', index=False)

    print(f'{model_label} observed control-treated gap by uplift decile:')
    display(gap_df)
    print()
    print(f'{model_label} cumulative uplift curve by decile:')
    display(curve_df)
    print()
    if curve_summary is not None:
        print(f'{model_label} uplift curve summary:')
        display(curve_summary)
        print()
    return gap_df, curve_df, curve_summary


observed_gap_xgboost, uplift_curve_xgboost, uplift_curve_summary_xgboost = save_observed_gap_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None:
    observed_gap_glmnet, uplift_curve_glmnet, uplift_curve_summary_glmnet = save_observed_gap_outputs(
        results_test_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    observed_gap_glmnet = None
    uplift_curve_glmnet = None
    uplift_curve_summary_glmnet = None
    print('Skipped GLMNET observed gap outputs because GLMNET scoring was not available.')




XGBoost observed control-treated gap by uplift decile:


,model,uplift_decile,n,treated_n,control_n,treated_events,control_events,observed_ed_rate,treated_observed_ed_rate,control_observed_ed_rate,...,observed_gap_ci_lower_95,observed_gap_ci_upper_95,avg_predicted_benefit,predicted_benefit_within_observed_gap_ci_95,predicted_minus_observed_gap,abs_predicted_minus_observed_gap,avg_pred_ed_if_treated,avg_pred_ed_if_control,treated_pct,ci_method
0,XGBoost,1,30,9,21,1,2,0.100000,0.111111,0.095238,...,-0.408481,0.269251,0.048251,True,0.064124,0.064124,0.120074,0.168325,0.300000,newcombe_wilson_difference_in_proportions
1,XGBoost,2,30,16,14,3,1,0.133333,0.187500,0.071429,...,-0.417367,0.248771,0.027164,True,0.143236,0.143236,0.119016,0.146180,0.533333,newcombe_wilson_difference_in_proportions
2,XGBoost,3,30,12,18,0,0,0.000000,0.000000,0.000000,...,-0.242494,0.175879,0.008736,True,0.008736,0.008736,0.114181,0.122917,0.400000,newcombe_wilson_difference_in_proportions
3,XGBoost,4,30,8,22,0,1,0.033333,0.000000,0.045455,...,-0.316338,0.217980,-0.002640,True,-0.048094,0.048094,0.113461,0.110822,0.266667,newcombe_wilson_difference_in_proportions
4,XGBoost,5,30,10,20,0,3,0.100000,0.000000,0.150000,...,-0.225164,0.360419,-0.007040,True,-0.157040,0.157040,0.112870,0.105830,0.333333,newcombe_wilson_difference_in_proportions
5,XGBoost,6,30,16,14,0,2,0.066667,0.000000,0.142857,...,-0.153514,0.399414,-0.010456,True,-0.153313,0.153313,0.112809,0.102353,0.533333,newcombe_wilson_difference_in_proportions
6,XGBoost,7,30,14,16,1,1,0.066667,0.071429,0.062500,...,-0.303568,0.270565,-0.014240,True,-0.005312,0.005312,0.114410,0.100170,0.466667,newcombe_wilson_difference_in_proportions
7,XGBoost,8,30,10,20,0,0,0.000000,0.000000,0.000000,...,-0.277533,0.161125,-0.018811,True,-0.018811,0.018811,0.113318,0.094506,0.333333,newcombe_wilson_difference_in_proportions
8,XGBoost,9,30,9,21,0,0,0.000000,0.000000,0.000000,...,-0.299145,0.154639,-0.024433,True,-0.024433,0.024433,0.118714,0.094281,0.300000,newcombe_wilson_difference_in_proportions
9,XGBoost,10,30,14,16,0,3,0.100000,0.000000,0.187500,...,-0.149395,0.430089,-0.033480,True,-0.220980,0.220980,0.125043,0.091563,0.466667,newcombe_wilson_difference_in_proportions



XGBoost cumulative uplift curve by decile:


,model,through_uplift_decile,population_fraction_targeted,n,treated_n,control_n,treated_events,control_events,avg_predicted_benefit,observed_control_minus_treated_gap,observed_gap_ci_lower_95,observed_gap_ci_upper_95,predicted_benefit_within_observed_gap_ci_95
0,XGBoost,1,0.1,30,9,21,1,2,0.048251,-0.015873,-0.408481,0.269251,True
1,XGBoost,2,0.2,60,25,35,4,3,0.037708,-0.074286,-0.316954,0.159758,True
2,XGBoost,3,0.3,90,37,53,4,3,0.028050,-0.051504,-0.227649,0.110850,True
3,XGBoost,4,0.4,120,45,75,4,4,0.020378,-0.035556,-0.186402,0.094149,True
4,XGBoost,5,0.5,150,55,95,4,7,0.014894,0.000957,-0.136450,0.115713,True
5,XGBoost,6,0.6,180,71,109,4,9,0.010669,0.026231,-0.092049,0.127388,True
6,XGBoost,7,0.7,210,85,125,5,10,0.007111,0.021176,-0.086381,0.115627,True
7,XGBoost,8,0.8,240,95,145,5,10,0.003870,0.016334,-0.079460,0.099603,True
8,XGBoost,9,0.9,270,104,166,5,10,0.000726,0.012164,-0.074595,0.086619,True
9,XGBoost,10,1.0,300,118,182,5,13,-0.002695,0.029056,-0.053154,0.100127,True



XGBoost uplift curve summary:


...

,model,uplift_decile,n,treated_n,control_n,treated_events,control_events,observed_ed_rate,treated_observed_ed_rate,control_observed_ed_rate,...,observed_gap_ci_lower_95,observed_gap_ci_upper_95,avg_predicted_benefit,predicted_benefit_within_observed_gap_ci_95,predicted_minus_observed_gap,abs_predicted_minus_observed_gap,avg_pred_ed_if_treated,avg_pred_ed_if_control,treated_pct,ci_method
0,GLMNET,1,30,11,19,1,1,0.066667,0.090909,0.052632,...,-0.368006,0.230155,0.044022,True,0.082299,0.082299,0.035710,0.079732,0.366667,newcombe_wilson_difference_in_proportions
1,GLMNET,2,30,13,17,0,0,0.000000,0.000000,0.000000,...,-0.228095,0.184318,0.040271,True,0.040271,0.040271,0.034523,0.074794,0.433333,newcombe_wilson_difference_in_proportions
2,GLMNET,3,30,11,19,0,0,0.000000,0.000000,0.000000,...,-0.258833,0.168179,0.038177,True,0.038177,0.038177,0.033556,0.071733,0.366667,newcombe_wilson_difference_in_proportions
3,GLMNET,4,30,11,19,0,1,0.033333,0.000000,0.052632,...,-0.249481,0.246387,0.036518,True,-0.016113,0.016113,0.033323,0.069842,0.366667,newcombe_wilson_difference_in_proportions
4,GLMNET,5,30,12,18,0,2,0.066667,0.000000,0.111111,...,-0.211474,0.327998,0.035405,True,-0.075706,0.075706,0.036684,0.072090,0.400000,newcombe_wilson_difference_in_proportions
5,GLMNET,6,30,10,20,0,1,0.033333,0.000000,0.050000,...,-0.268651,0.236131,0.034239,True,-0.015761,0.015761,0.037637,0.071876,0.333333,newcombe_wilson_difference_in_proportions
6,GLMNET,7,30,17,13,0,2,0.066667,0.000000,0.153846,...,-0.141060,0.422346,0.032882,True,-0.120964,0.120964,0.037847,0.070729,0.566667,newcombe_wilson_difference_in_proportions
7,GLMNET,8,30,12,18,0,4,0.133333,0.000000,0.222222,...,-0.152485,0.452146,0.031156,True,-0.191066,0.191066,0.041132,0.072288,0.400000,newcombe_wilson_difference_in_proportions
8,GLMNET,9,30,10,20,0,0,0.000000,0.000000,0.000000,...,-0.277533,0.161125,0.028667,True,0.028667,0.028667,0.044195,0.072862,0.333333,newcombe_wilson_difference_in_proportions
9,GLMNET,10,30,11,19,4,2,0.200000,0.363636,0.105263,...,-0.616840,0.162276,0.023024,True,0.281397,0.281397,0.058430,0.081454,0.366667,newcombe_wilson_difference_in_proportions



GLMNET cumulative uplift curve by decile:


,model,through_uplift_decile,population_fraction_targeted,n,treated_n,control_n,treated_events,control_events,avg_predicted_benefit,observed_control_minus_treated_gap,observed_gap_ci_lower_95,observed_gap_ci_upper_95,predicted_benefit_within_observed_gap_ci_95
0,GLMNET,1,0.1,30,11,19,1,1,0.044022,-0.038278,-0.368006,0.230155,True
1,GLMNET,2,0.2,60,24,36,1,1,0.042146,-0.013889,-0.197498,0.134304,True
2,GLMNET,3,0.3,90,35,55,1,1,0.040823,-0.010390,-0.142114,0.090996,True
3,GLMNET,4,0.4,120,46,74,1,2,0.039747,0.005288,-0.105910,0.089445,True
4,GLMNET,5,0.5,150,58,92,1,4,0.038879,0.026237,-0.074373,0.103467,True
5,GLMNET,6,0.6,180,68,112,1,5,0.038105,0.029937,-0.059492,0.097668,True
6,GLMNET,7,0.7,210,85,125,1,7,0.037359,0.044235,-0.036283,0.109008,True
7,GLMNET,8,0.8,240,97,143,1,11,0.036584,0.066614,-0.012612,0.130667,True
8,GLMNET,9,0.9,270,107,163,1,11,0.035704,0.058139,-0.012952,0.115138,True
9,GLMNET,10,1.0,300,118,182,5,13,0.034436,0.029056,-0.053154,0.100127,True



GLMNET uplift curve summary:


,model,observed_gap_area_under_curve,predicted_benefit_area_under_curve,final_cumulative_observed_gap,final_cumulative_avg_predicted_benefit
0,GLMNET,0.020156,0.034858,0.029056,0.034436


---
## 17. VARIABLE IMPORTANCE
---


In [31]:
importance_treated_xgboost = xgb_importance_frame(model_treated)
importance_control_xgboost = xgb_importance_frame(model_control)

# Backward-compatible aliases for the primary XGBoost importance tables.
importance_treated = importance_treated_xgboost
importance_control = importance_control_xgboost

print('Top variables in XGBoost treated model:')
display(importance_treated_xgboost.head(20))
print()

print('Top variables in XGBoost control model:')
display(importance_control_xgboost.head(20))
print()

if enet_treated is not None and enet_control is not None:
    glmnet_importance_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_importance_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')

    print('Top variables in GLMNET treated model:')
    display(glmnet_importance_treated.head(20))
    print()

    print('Top variables in GLMNET control model:')
    display(glmnet_importance_control.head(20))
    print()
else:
    glmnet_importance_treated = None
    glmnet_importance_control = None


Top variables in XGBoost treated model:


,feature,gain,cover,frequency
0,percolator_utilization_score,8.274089,24.443964,3.0
1,total_cost_last_6m,7.498248,24.704767,1.0
2,ed_visits_last_30d,3.896444,2.234379,1.0
3,admits_last_6m,3.747496,25.684122,1.0
4,current_risk_score,3.678734,18.268389,3.0
5,case_manager_name_CM_09,3.207897,23.640059,4.0
6,chf_flag,2.102352,23.793196,2.0
7,age,1.691691,3.438812,2.0
8,rx_count_last_6m,1.655909,10.501611,2.0
9,county_County_E,1.534343,21.593449,5.0



Top variables in XGBoost control model:


,feature,gain,cover,frequency
0,percolator_clinical_score,6.991149,38.609993,9.0
1,percolator_utilization_score,4.992712,36.094261,2.0
2,specialist_visits_last_6m,4.624862,14.248364,1.0
3,ed_visits_last_6m,4.491296,11.933380,1.0
4,current_risk_score,4.413888,31.979582,7.0
5,age,3.490537,31.101122,3.0
6,pcp_visits_last_6m,3.476344,31.206755,3.0
7,total_cost_last_6m,1.834459,25.320747,1.0
8,county_County_B,1.654583,28.198338,1.0
9,percolator_sdoh_score,1.460143,31.629393,3.0



Top variables in GLMNET treated model:


,feature,mean_abs_model_contribution,coefficient,model,importance_type
0,percolator_utilization_score,0.037785,0.048912,Treated Model,standardized_logit_contribution
1,total_cost_last_6m,0.034909,0.042499,Treated Model,standardized_logit_contribution
2,admits_last_6m,0.028960,0.035669,Treated Model,standardized_logit_contribution
3,current_risk_score,0.027279,0.033101,Treated Model,standardized_logit_contribution
4,behavioral_health_risk_flag,0.024967,0.025587,Treated Model,standardized_logit_contribution
5,chf_flag,0.022911,0.032081,Treated Model,standardized_logit_contribution
6,food_insecurity_flag,0.022603,0.027565,Treated Model,standardized_logit_contribution
7,ed_visits_last_6m,0.022007,0.024613,Treated Model,standardized_logit_contribution
8,service_region_Central,0.020065,0.024098,Treated Model,standardized_logit_contribution
9,county_County_E,0.019762,0.024971,Treated Model,standardized_logit_contribution



Top variables in GLMNET control model:


,feature,mean_abs_model_contribution,coefficient,model,importance_type
0,current_risk_score,0.032714,0.039696,Control Model,standardized_logit_contribution
1,percolator_clinical_score,0.032144,0.043194,Control Model,standardized_logit_contribution
2,admits_last_6m,0.023513,0.028960,Control Model,standardized_logit_contribution
3,chf_flag,0.020166,0.028237,Control Model,standardized_logit_contribution
4,ed_visits_last_6m,0.017245,0.019287,Control Model,standardized_logit_contribution
5,risk_tier_Very_High,0.001677,0.031331,Control Model,standardized_logit_contribution
6,percolator_utilization_score,0.000873,0.001130,Control Model,standardized_logit_contribution
7,risk_tier_High,0.000255,0.000363,Control Model,standardized_logit_contribution
8,language_Spanish,0.000000,0.000000,Control Model,standardized_logit_contribution
9,case_manager_name_CM_14,0.000000,0.000000,Control Model,standardized_logit_contribution


---
## 18. SCORE FULL FILES
---


In [32]:
full_x_df = model_df[feature_cols].copy()
_, [full_matrix_raw] = make_design_matrix([full_x_df])
full_matrix = align_to_columns(full_matrix_raw, combined_matrix.columns)

full_pred_treated_xgboost = model_treated.predict(make_dmatrix(full_matrix))
full_pred_control_xgboost = model_control.predict(make_dmatrix(full_matrix))
scored_full_xgboost = build_uplift_results(model_df, full_pred_treated_xgboost, full_pred_control_xgboost)

# Backward-compatible alias for the primary XGBoost scored file.
scored_full = scored_full_xgboost

if enet_treated is not None and enet_control is not None:
    full_pred_treated_glmnet = enet_treated['best_model'].predict_proba(full_matrix)[:, 1]
    full_pred_control_glmnet = enet_control['best_model'].predict_proba(full_matrix)[:, 1]
    scored_full_glmnet = build_uplift_results(model_df, full_pred_treated_glmnet, full_pred_control_glmnet)
else:
    full_pred_treated_glmnet = None
    full_pred_control_glmnet = None
    scored_full_glmnet = None


---
## 19. WRITE OUTPUTS
---


In [33]:
scored_full_xgboost.to_csv(xgboost_output_path, index=False)
decile_summary_xgboost.to_csv(xgboost_summary_path, index=False)

print('XGBoost scored full file written to:', xgboost_output_path, '\n')
print('XGBoost decile summary written to:', xgboost_summary_path, '\n')

if scored_full_glmnet is not None and decile_summary_glmnet is not None:
    scored_full_glmnet.to_csv(glmnet_output_path, index=False)
    decile_summary_glmnet.to_csv(glmnet_summary_path, index=False)

    print('GLMNET scored full file written to:', glmnet_output_path, '\n')
    print('GLMNET decile summary written to:', glmnet_summary_path, '\n')
else:
    print('Skipped GLMNET core CSV outputs because GLMNET scoring was not available.\n')


XGBoost scored full file written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/uplift_scored_output.csv 

XGBoost decile summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/uplift_decile_summary.csv 

GLMNET scored full file written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet/uplift_scored_output.csv 

GLMNET decile summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet/uplift_decile_summary.csv 



---
## 20. INTERPRETATION
---


In [34]:
print('INTERPRETATION:')
print('- pred_ed_if_treated = predicted probability of ED within 90d if treated')
print('- pred_ed_if_control = predicted probability of ED within 90d if not treated')
print('- benefit_score = pred_ed_if_control - pred_ed_if_treated')
print('- Higher benefit_score means treatment is predicted to reduce ED risk more')
print('- Uplift decile 1 = highest predicted treatment benefit')


INTERPRETATION:
- pred_ed_if_treated = predicted probability of ED within 90d if treated
- pred_ed_if_control = predicted probability of ED within 90d if not treated
- benefit_score = pred_ed_if_control - pred_ed_if_treated
- Higher benefit_score means treatment is predicted to reduce ED risk more
- Uplift decile 1 = highest predicted treatment benefit


---
## DASHBOARD VIEWS
---


In [35]:
dashboard_folder = xgboost_output_folder


def save_bar_chart(df, x_col, y_col, title, x_label, y_label, path, width=8, height=5, y_max=None):
    fig, ax = plt.subplots(figsize=(width, height))
    ax.bar(df[x_col].astype(str), df[y_col])
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    if y_max is not None:
        ax.set_ylim(0, y_max)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def save_decile_dashboard_charts(decile_df, folder, model_label, benefit_y_max=None):
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'avg_benefit_score',
        f'{model_label}: Average Predicted Intervention Benefit by Uplift Decile',
        'Uplift Decile: 1 = Highest Predicted Benefit',
        'Average Benefit Score',
        folder / 'dashboard_avg_benefit_by_decile.png',
        y_max=benefit_y_max,
    )
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'observed_ed_rate',
        f'{model_label}: Observed 90-Day ED Rate by Uplift Decile',
        'Uplift Decile',
        'Observed ED Rate',
        folder / 'dashboard_observed_ed_rate_by_decile.png',
    )
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'treated_pct',
        f'{model_label}: Current Treatment Penetration by Uplift Decile',
        'Uplift Decile',
        'Percent Treated',
        folder / 'dashboard_treated_pct_by_decile.png',
    )

    decile_long = decile_df.melt(
        id_vars='uplift_decile',
        value_vars=['avg_pred_ed_if_treated', 'avg_pred_ed_if_control'],
        var_name='scenario',
        value_name='predicted_ed_rate',
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    scenarios = list(decile_long['scenario'].unique())
    x_values = np.arange(len(decile_df))
    bar_width = 0.35
    for offset, scenario in enumerate(scenarios):
        values = decile_long[decile_long['scenario'] == scenario]['predicted_ed_rate'].to_numpy()
        ax.bar(x_values + (offset - 0.5) * bar_width, values, width=bar_width, label=scenario)
    ax.set_xticks(x_values)
    ax.set_xticklabels(decile_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Predicted ED Risk Treated vs Control by Decile')
    ax.set_xlabel('Uplift Decile')
    ax.set_ylabel('Predicted ED Rate')
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_predicted_treated_vs_control.png', dpi=150)
    plt.close(fig)


save_decile_dashboard_charts(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')
print('XGBoost dashboard charts saved to:', xgboost_output_folder)

if decile_summary_glmnet is not None:
    save_decile_dashboard_charts(decile_summary_glmnet, glmnet_output_folder, 'GLMNET')
    print('GLMNET dashboard charts saved to:', glmnet_output_folder)
else:
    print('Skipped GLMNET dashboard charts because GLMNET scoring was not available.')


XGBoost dashboard charts saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost


GLMNET dashboard charts saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet


---
## ROI PER DECILE
---


In [36]:
cost_per_ed_visit = 1200
cost_per_intervention = 250


def build_roi_summary(decile_df):
    roi_df = decile_df.copy()
    roi_df['expected_ed_rate_reduction'] = roi_df['avg_benefit_score']
    roi_df['expected_ed_visits_avoided'] = roi_df['n'] * roi_df['expected_ed_rate_reduction']
    roi_df['gross_savings'] = roi_df['expected_ed_visits_avoided'] * cost_per_ed_visit
    roi_df['intervention_cost'] = roi_df['n'] * cost_per_intervention
    roi_df['net_savings'] = roi_df['gross_savings'] - roi_df['intervention_cost']
    roi_df['roi'] = roi_df['net_savings'] / roi_df['intervention_cost']
    return roi_df


def save_roi_outputs(decile_df, folder, model_label):
    roi_df = build_roi_summary(decile_df)
    print(f'{model_label} ROI summary:')
    display(roi_df)
    roi_df.to_csv(folder / 'uplift_roi_by_decile.csv', index=False)
    save_bar_chart(
        roi_df,
        'uplift_decile',
        'net_savings',
        f'{model_label}: Estimated Net Savings by Uplift Decile',
        'Uplift Decile',
        'Estimated Net Savings',
        folder / 'dashboard_roi_net_savings_by_decile.png',
    )
    print(f'{model_label} ROI summary saved to:', folder)
    print()
    return roi_df


roi_summary_xgboost = save_roi_outputs(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')

# Backward-compatible alias for the primary XGBoost ROI summary.
roi_summary = roi_summary_xgboost

if decile_summary_glmnet is not None:
    roi_summary_glmnet = save_roi_outputs(decile_summary_glmnet, glmnet_output_folder, 'GLMNET')
else:
    roi_summary_glmnet = None
    print('Skipped GLMNET ROI outputs because GLMNET scoring was not available.')


XGBoost ROI summary:


,uplift_decile,n,avg_benefit_score,observed_ed_rate,treated_pct,avg_pred_ed_if_treated,avg_pred_ed_if_control,expected_ed_rate_reduction,expected_ed_visits_avoided,gross_savings,intervention_cost,net_savings,roi
0,1,30,0.048251,0.100000,0.300000,0.120074,0.168325,0.048251,1.447529,1737.034500,7500,-5762.965500,-0.768395
1,2,30,0.027164,0.133333,0.533333,0.119016,0.146180,0.027164,0.814929,977.914497,7500,-6522.085503,-0.869611
2,3,30,0.008736,0.000000,0.400000,0.114181,0.122917,0.008736,0.262068,314.481564,7500,-7185.518436,-0.958069
3,4,30,-0.002640,0.033333,0.266667,0.113461,0.110822,-0.002640,-0.079186,-95.023457,7500,-7595.023457,-1.012670
4,5,30,-0.007040,0.100000,0.333333,0.112870,0.105830,-0.007040,-0.211211,-253.453128,7500,-7753.453128,-1.033794
5,6,30,-0.010456,0.066667,0.533333,0.112809,0.102353,-0.010456,-0.313673,-376.407500,7500,-7876.407500,-1.050188
6,7,30,-0.014240,0.066667,0.466667,0.114410,0.100170,-0.014240,-0.427211,-512.653593,7500,-8012.653593,-1.068354
7,8,30,-0.018811,0.000000,0.333333,0.113318,0.094506,-0.018811,-0.564340,-677.207686,7500,-8177.207686,-1.090294
8,9,30,-0.024433,0.000000,0.300000,0.118714,0.094281,-0.024433,-0.733004,-879.605368,7500,-8379.605368,-1.117281
9,10,30,-0.033480,0.100000,0.466667,0.125043,0.091563,-0.033480,-1.004388,-1205.265507,7500,-8705.265507,-1.160702


XGBoost ROI summary saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost

GLMNET ROI summary:


,uplift_decile,n,avg_benefit_score,observed_ed_rate,treated_pct,avg_pred_ed_if_treated,avg_pred_ed_if_control,expected_ed_rate_reduction,expected_ed_visits_avoided,gross_savings,intervention_cost,net_savings,roi
0,1,30,0.044022,0.066667,0.366667,0.035710,0.079732,0.044022,1.320656,1584.787195,7500,-5915.212805,-0.788695
1,2,30,0.040271,0.000000,0.433333,0.034523,0.074794,0.040271,1.208127,1449.752012,7500,-6050.247988,-0.806700
2,3,30,0.038177,0.000000,0.366667,0.033556,0.071733,0.038177,1.145314,1374.377278,7500,-6125.622722,-0.816750
3,4,30,0.036518,0.033333,0.366667,0.033323,0.069842,0.036518,1.095548,1314.657562,7500,-6185.342438,-0.824712
4,5,30,0.035405,0.066667,0.400000,0.036684,0.072090,0.035405,1.062164,1274.596241,7500,-6225.403759,-0.830054
5,6,30,0.034239,0.033333,0.333333,0.037637,0.071876,0.034239,1.027168,1232.601768,7500,-6267.398232,-0.835653
6,7,30,0.032882,0.066667,0.566667,0.037847,0.070729,0.032882,0.986456,1183.747350,7500,-6316.252650,-0.842167
7,8,30,0.031156,0.133333,0.400000,0.041132,0.072288,0.031156,0.934673,1121.607287,7500,-6378.392713,-0.850452
8,9,30,0.028667,0.000000,0.333333,0.044195,0.072862,0.028667,0.860009,1032.010668,7500,-6467.989332,-0.862399
9,10,30,0.023024,0.200000,0.366667,0.058430,0.081454,0.023024,0.690705,828.846411,7500,-6671.153589,-0.889487


GLMNET ROI summary saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet



---
## WRITE-UP TOP BENEFIT DECILE SUMMARY
---


In [37]:
def top_benefit_decile_summary(results, roi_df, model_label):
    if results is None or roi_df is None:
        return None

    top_decile = results[results['uplift_decile'] == 1].copy()
    if top_decile.empty:
        return None

    treated = top_decile[top_decile['intervention_flag'] == 1]
    control = top_decile[top_decile['intervention_flag'] == 0]
    treated_rate = treated['outcome_ed_90d'].mean() if not treated.empty else np.nan
    control_rate = control['outcome_ed_90d'].mean() if not control.empty else np.nan
    top_roi = roi_df.loc[roi_df['uplift_decile'] == 1].iloc[0]

    return pd.DataFrame(
        [
            {
                'model': model_label,
                'top_decile_n': len(top_decile),
                'top_decile_treated_n': len(treated),
                'top_decile_control_n': len(control),
                'top_decile_avg_predicted_benefit': top_decile['benefit_score'].mean(),
                'top_decile_observed_ed_rate': top_decile['outcome_ed_90d'].mean(),
                'top_decile_treated_observed_ed_rate': treated_rate,
                'top_decile_control_observed_ed_rate': control_rate,
                'top_decile_observed_control_minus_treated_gap': control_rate - treated_rate,
                'top_decile_treated_pct': top_decile['intervention_flag'].mean(),
                'top_decile_estimated_ed_visits_avoided': top_roi['expected_ed_visits_avoided'],
                'top_decile_gross_savings': top_roi['gross_savings'],
                'top_decile_intervention_cost': top_roi['intervention_cost'],
                'top_decile_net_savings': top_roi['net_savings'],
                'top_decile_roi': top_roi['roi'],
            }
        ]
    )


def save_top_decile_summary(results, roi_df, folder, model_label):
    summary = top_benefit_decile_summary(results, roi_df, model_label)
    if summary is None:
        return None
    summary.to_csv(folder / 'top_benefit_decile_summary.csv', index=False)
    print(f'{model_label} top benefit decile summary:')
    display(summary)
    print()
    return summary


top_decile_summary_xgboost = save_top_decile_summary(
    results_test_xgboost,
    roi_summary_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None and roi_summary_glmnet is not None:
    top_decile_summary_glmnet = save_top_decile_summary(
        results_test_glmnet,
        roi_summary_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    top_decile_summary_glmnet = None
    print('Skipped GLMNET top benefit decile summary because GLMNET scoring or ROI was not available.')


XGBoost top benefit decile summary:


,model,top_decile_n,top_decile_treated_n,top_decile_control_n,top_decile_avg_predicted_benefit,top_decile_observed_ed_rate,top_decile_treated_observed_ed_rate,top_decile_control_observed_ed_rate,top_decile_observed_control_minus_treated_gap,top_decile_treated_pct,top_decile_estimated_ed_visits_avoided,top_decile_gross_savings,top_decile_intervention_cost,top_decile_net_savings,top_decile_roi
0,XGBoost,30,9,21,0.048251,0.1,0.111111,0.095238,-0.015873,0.3,1.447529,1737.0345,7500.0,-5762.9655,-0.768395



GLMNET top benefit decile summary:


,model,top_decile_n,top_decile_treated_n,top_decile_control_n,top_decile_avg_predicted_benefit,top_decile_observed_ed_rate,top_decile_treated_observed_ed_rate,top_decile_control_observed_ed_rate,top_decile_observed_control_minus_treated_gap,top_decile_treated_pct,top_decile_estimated_ed_visits_avoided,top_decile_gross_savings,top_decile_intervention_cost,top_decile_net_savings,top_decile_roi
0,GLMNET,30,11,19,0.044022,0.066667,0.090909,0.052632,-0.038278,0.366667,1.320656,1584.787195,7500.0,-5915.212805,-0.788695


---
## RISK MODEL DRIVER OUTPUTS
---


In [38]:
def save_model_driver_outputs(
    treated_importance,
    control_importance,
    folder,
    model_label,
    value_col,
    x_label,
    csv_name='shap_importance_treated_control_models.csv',
):
    combined = pd.concat([treated_importance, control_importance], ignore_index=True)
    combined.to_csv(folder / csv_name, index=False)

    for driver_df, treatment_label, filename in [
        (treated_importance, 'Treated Model', 'dashboard_shap_treated_model.png'),
        (control_importance, 'Control Model', 'dashboard_shap_control_model.png'),
    ]:
        top = driver_df.nlargest(20, value_col).sort_values(value_col)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(top['feature'], top[value_col])
        ax.set_title(f'{model_label}: Top Risk Drivers - {treatment_label}')
        ax.set_xlabel(x_label)
        ax.set_ylabel('Feature')
        fig.tight_layout()
        fig.savefig(folder / filename, dpi=150)
        plt.close(fig)

    print(f'{model_label} risk-driver outputs saved to:', folder)
    return combined


shap_treated_importance = shap_importance_frame(model_treated, x_test, 'Treated Model')
shap_control_importance = shap_importance_frame(model_control, x_test, 'Control Model')
shap_importance_combined = save_model_driver_outputs(
    shap_treated_importance,
    shap_control_importance,
    xgboost_output_folder,
    'XGBoost',
    'mean_abs_shap',
    'Mean Absolute SHAP Contribution',
)

if enet_treated is not None and enet_control is not None:
    # GLMNET does not use XGBoost SHAP contributions here, so save an analogous
    # standardized logit-contribution driver artifact in the GLMNET folder.
    glmnet_driver_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_driver_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')
    glmnet_driver_combined = save_model_driver_outputs(
        glmnet_driver_treated,
        glmnet_driver_control,
        glmnet_output_folder,
        'GLMNET',
        'mean_abs_model_contribution',
        'Mean Absolute Standardized Logit Contribution',
    )
else:
    glmnet_driver_treated = None
    glmnet_driver_control = None
    glmnet_driver_combined = None
    print('Skipped GLMNET model-driver outputs because GLMNET scoring was not available.')

print('\nFiles currently in XGBoost output folder:')
print('\n'.join(str(path) for path in sorted(xgboost_output_folder.iterdir())))

print('\nFiles currently in GLMNET output folder:')
print('\n'.join(str(path) for path in sorted(glmnet_output_folder.iterdir())))


XGBoost risk-driver outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost


GLMNET risk-driver outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet

Files currently in XGBoost output folder:
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/.ipynb_checkpoints
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/calibration_by_decile.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/calibration_summary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/dashboard_avg_benefit_by_decile.png
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/dashboard_calibration_plot.png
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/dashboard_observed_ed_rate_by_decile.png
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/dashboard_observed_gap_by_decile.png
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost/dashboard_predicted_treated_vs_control.png
/home/sagemaker-

---
## BENEFIT SCORE DRIVER IMPORTANCE
---


In [39]:
def save_benefit_driver_outputs(importance_df, folder, model_label, value_col, x_label):
    importance_df.to_csv(
        folder / 'shap_importance_benefit_score.csv',
        index=False,
    )

    top = importance_df.head(20).sort_values(value_col)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top[value_col])
    ax.set_title(f'{model_label}: Top Drivers of Predicted Treatment Benefit')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_shap_benefit_score.png', dpi=150)
    plt.close(fig)

    print(f'{model_label} benefit-driver outputs saved to:', folder)
    print()


def xgboost_benefit_shap_importance_frame(model_treated, model_control, x_matrix):
    benefit_dtest = make_dmatrix(x_matrix)

    control_contribs = model_control.predict(benefit_dtest, pred_contribs=True)
    treated_contribs = model_treated.predict(benefit_dtest, pred_contribs=True)
    shap_columns = [*x_matrix.columns, 'BIAS']

    control_shap = pd.DataFrame(control_contribs, columns=shap_columns, index=x_matrix.index)
    treated_shap = pd.DataFrame(treated_contribs, columns=shap_columns, index=x_matrix.index)

    # Signed contribution to model-estimated benefit on the XGBoost raw-margin scale.
    benefit_shap = control_shap - treated_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')

    return (
        pd.DataFrame(
            {
                'feature': benefit_shap_no_bias.columns,
                'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
                'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
                'importance_type': 'xgboost_raw_margin_shap_difference',
            }
        )
        .sort_values('mean_abs_benefit_shap', ascending=False)
        .reset_index(drop=True)
    )


def glmnet_benefit_contribution_importance_frame(treated_model_info, control_model_info, x_matrix):
    treated_pipeline = treated_model_info['best_model']
    control_pipeline = control_model_info['best_model']

    treated_scaler = treated_pipeline.named_steps['standardscaler']
    control_scaler = control_pipeline.named_steps['standardscaler']
    treated_model = treated_pipeline.named_steps['logisticregressioncv']
    control_model = control_pipeline.named_steps['logisticregressioncv']

    treated_contrib = pd.DataFrame(
        treated_scaler.transform(x_matrix) * treated_model.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )
    control_contrib = pd.DataFrame(
        control_scaler.transform(x_matrix) * control_model.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )

    # Signed contribution to model-estimated benefit on the GLMNET logit scale.
    benefit_contrib = control_contrib - treated_contrib

    return (
        pd.DataFrame(
            {
                'feature': benefit_contrib.columns,
                'mean_abs_benefit_contribution': benefit_contrib.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_contribution': benefit_contrib.mean(axis=0).to_numpy(),
                'pct_positive_benefit_contribution': benefit_contrib.gt(0).mean(axis=0).to_numpy(),
                'control_coefficient': control_model.coef_.ravel(),
                'treated_coefficient': treated_model.coef_.ravel(),
                'coefficient_difference': control_model.coef_.ravel() - treated_model.coef_.ravel(),
                'importance_type': 'glmnet_standardized_logit_contribution_difference',
            }
        )
        .sort_values('mean_abs_benefit_contribution', ascending=False)
        .reset_index(drop=True)
    )


xgboost_benefit_shap_importance = xgboost_benefit_shap_importance_frame(
    model_treated,
    model_control,
    x_test,
)

print('Top XGBoost SHAP features driving predicted treatment benefit:')
display(xgboost_benefit_shap_importance.head(20))

save_benefit_driver_outputs(
    xgboost_benefit_shap_importance,
    xgboost_output_folder,
    'XGBoost',
    'mean_abs_benefit_shap',
    'Mean Absolute SHAP Contribution to Benefit',
)

if enet_treated is not None and enet_control is not None:
    glmnet_benefit_importance = glmnet_benefit_contribution_importance_frame(
        enet_treated,
        enet_control,
        x_test,
    )

    print('Top GLMNET features driving predicted treatment benefit:')
    display(glmnet_benefit_importance.head(20))

    save_benefit_driver_outputs(
        glmnet_benefit_importance,
        glmnet_output_folder,
        'GLMNET',
        'mean_abs_benefit_contribution',
        'Mean Absolute Standardized Logit Contribution to Benefit',
    )
else:
    glmnet_benefit_importance = None
    print('Skipped GLMNET benefit-driver outputs because GLMNET scoring was not available.')


Top XGBoost SHAP features driving predicted treatment benefit:


,feature,mean_abs_benefit_shap,mean_signed_benefit_shap,pct_positive_benefit_shap,importance_type
0,percolator_clinical_score,0.133730,-0.002423,0.213333,xgboost_raw_margin_shap_difference
1,current_risk_score,0.054613,0.003737,0.390000,xgboost_raw_margin_shap_difference
2,age,0.033765,-0.001946,0.200000,xgboost_raw_margin_shap_difference
3,county_County_E,0.031976,-0.000349,0.806667,xgboost_raw_margin_shap_difference
4,pcp_visits_last_6m,0.030352,-0.002270,0.366667,xgboost_raw_margin_shap_difference
5,percolator_sdoh_score,0.021474,-0.000486,0.243333,xgboost_raw_margin_shap_difference
6,language_English,0.021012,-0.002309,0.243333,xgboost_raw_margin_shap_difference
7,percolator_utilization_score,0.020401,0.004240,0.216667,xgboost_raw_margin_shap_difference
8,behavioral_health_risk_flag,0.019603,-0.001435,0.386667,xgboost_raw_margin_shap_difference
9,case_manager_name_CM_09,0.018550,0.002090,0.933333,xgboost_raw_margin_shap_difference


XGBoost benefit-driver outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/XGBoost

Top GLMNET features driving predicted treatment benefit:


,feature,mean_abs_benefit_contribution,mean_signed_benefit_contribution,pct_positive_benefit_contribution,control_coefficient,treated_coefficient,coefficient_difference,importance_type
0,percolator_utilization_score,0.036912,-0.001980,0.513333,0.001130,0.048912,-0.047782,glmnet_standardized_logit_contribution_difference
1,total_cost_last_6m,0.034909,-0.004155,0.540000,0.000000,0.042499,-0.042499,glmnet_standardized_logit_contribution_difference
2,percolator_clinical_score,0.025912,0.000111,0.443333,0.043194,0.008374,0.034820,glmnet_standardized_logit_contribution_difference
3,behavioral_health_risk_flag,0.024967,0.001508,0.613333,0.000000,0.025587,-0.025587,glmnet_standardized_logit_contribution_difference
4,food_insecurity_flag,0.022603,0.000909,0.786667,0.000000,0.027565,-0.027565,glmnet_standardized_logit_contribution_difference
5,service_region_Central,0.020065,-0.002893,0.783333,0.000000,0.024098,-0.024098,glmnet_standardized_logit_contribution_difference
6,county_County_E,0.019762,-0.001254,0.806667,0.000000,0.024971,-0.024971,glmnet_standardized_logit_contribution_difference
7,risk_tier_High,0.018756,-0.002838,0.860000,0.000363,0.027079,-0.026716,glmnet_standardized_logit_contribution_difference
8,asthma_flag,0.018506,-0.000126,0.166667,0.000000,-0.024828,0.024828,glmnet_standardized_logit_contribution_difference
9,program_Complex_CM,0.018302,0.000826,0.213333,0.000000,-0.022315,0.022315,glmnet_standardized_logit_contribution_difference


GLMNET benefit-driver outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/T-Learner/GLMNet



---
## X-LEARNER BENEFIT SCORE DRIVER IMPORTANCE
---

This section creates GLMNet X-learner benefit-driver outputs so Task 6 can compare the T-learner and X-learner benefit-driver patterns side by side.


In [40]:

def glmnet_xlearner_benefit_contribution_importance_frame(tau_treated_model, tau_control_model, propensity_model, x_matrix):
    treated_scaler = tau_treated_model.named_steps['standardscaler']
    control_scaler = tau_control_model.named_steps['standardscaler']
    if treated_scaler is not control_scaler:
        raise ValueError('GLMNET X-learner treated/control effect models must share the same scaler for contribution attribution.')
    shared_x = treated_scaler.transform(x_matrix)
    treated_reg = tau_treated_model.named_steps['elasticnetcv']
    control_reg = tau_control_model.named_steps['elasticnetcv']

    propensity = clipped_propensity(propensity_model, x_matrix)
    treated_weight = 1 - propensity
    control_weight = propensity

    treated_effect_contrib = pd.DataFrame(
        shared_x * treated_reg.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )
    control_effect_contrib = pd.DataFrame(
        shared_x * control_reg.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )

    weighted_benefit_contrib = treated_effect_contrib.mul(treated_weight, axis=0) + control_effect_contrib.mul(control_weight, axis=0)

    return (
        pd.DataFrame(
            {
                'feature': weighted_benefit_contrib.columns,
                'mean_abs_xlearner_benefit_contribution': weighted_benefit_contrib.abs().mean(axis=0).to_numpy(),
                'mean_signed_xlearner_benefit_contribution': weighted_benefit_contrib.mean(axis=0).to_numpy(),
                'pct_positive_xlearner_benefit_contribution': weighted_benefit_contrib.gt(0).mean(axis=0).to_numpy(),
                'treated_effect_coefficient': treated_reg.coef_.ravel(),
                'control_effect_coefficient': control_reg.coef_.ravel(),
                'avg_treated_effect_weight': treated_weight.mean(),
                'avg_control_effect_weight': control_weight.mean(),
                'importance_type': 'glmnet_xlearner_weighted_shared_standardized_effect_contribution',
            }
        )
        .sort_values('mean_abs_xlearner_benefit_contribution', ascending=False)
        .reset_index(drop=True)
    )


def save_xlearner_glmnet_benefit_driver_outputs(importance_df, folder):
    importance_path = folder / 'xlearner_benefit_driver_importance.csv'
    importance_df.to_csv(importance_path, index=False)

    # Also save to the common benefit-driver filename so README generation can use parallel paths.
    importance_df.to_csv(folder / 'shap_importance_benefit_score.csv', index=False)

    top = importance_df.head(20).sort_values('mean_abs_xlearner_benefit_contribution')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_xlearner_benefit_contribution'])
    ax.set_title('GLMNET X-Learner: Top Drivers of Predicted Treatment Benefit')
    ax.set_xlabel('Mean Absolute Weighted Standardized Effect Contribution')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_xlearner_benefit_drivers.png', dpi=150)
    fig.savefig(folder / 'dashboard_shap_benefit_score.png', dpi=150)
    plt.close(fig)

    print('GLMNET X-learner benefit-driver outputs saved to:', folder)
    print('X-learner benefit-driver CSV:', importance_path)


def save_t_vs_x_benefit_driver_comparison_chart(t_importance, x_importance, folder):
    if t_importance is None or x_importance is None:
        print('Skipped T-learner versus X-learner benefit-driver comparison chart because one input is missing.')
        return None

    t_top = t_importance.head(15).sort_values('mean_abs_benefit_contribution')
    x_top = x_importance.head(15).sort_values('mean_abs_xlearner_benefit_contribution')

    fig, axes = plt.subplots(ncols=2, figsize=(14, 6))
    axes[0].barh(t_top['feature'], t_top['mean_abs_benefit_contribution'])
    axes[0].set_title('GLMNET T-Learner')
    axes[0].set_xlabel('Mean Absolute Benefit Contribution')
    axes[0].set_ylabel('Feature')

    axes[1].barh(x_top['feature'], x_top['mean_abs_xlearner_benefit_contribution'])
    axes[1].set_title('GLMNET X-Learner')
    axes[1].set_xlabel('Mean Absolute Benefit Contribution')
    axes[1].set_ylabel('Feature')

    fig.suptitle('Top Drivers of Predicted Treatment Benefit')
    fig.tight_layout()
    chart_path = folder / 'dashboard_t_vs_x_benefit_driver_comparison.png'
    fig.savefig(chart_path, dpi=150)
    plt.close(fig)
    print('T-learner versus X-learner benefit-driver comparison chart saved to:', chart_path)
    return chart_path


if (
    'glmnet_tau_treated_model' in globals()
    and 'glmnet_tau_control_model' in globals()
    and glmnet_tau_treated_model is not None
    and glmnet_tau_control_model is not None
):
    glmnet_xlearner_benefit_importance = glmnet_xlearner_benefit_contribution_importance_frame(
        glmnet_tau_treated_model,
        glmnet_tau_control_model,
        propensity_model,
        x_test,
    )
    print('Top GLMNET X-learner features driving predicted treatment benefit:')
    display(glmnet_xlearner_benefit_importance.head(20))

    save_xlearner_glmnet_benefit_driver_outputs(
        glmnet_xlearner_benefit_importance,
        xlearner_glmnet_output_folder,
    )
    save_t_vs_x_benefit_driver_comparison_chart(
        glmnet_benefit_importance,
        glmnet_xlearner_benefit_importance,
        xlearner_glmnet_output_folder,
    )
else:
    glmnet_xlearner_benefit_importance = None
    print('Skipped GLMNET X-learner benefit-driver outputs because GLMNET X-learner models were not available.')


Top GLMNET X-learner features driving predicted treatment benefit:


,feature,mean_abs_xlearner_benefit_contribution,mean_signed_xlearner_benefit_contribution,pct_positive_xlearner_benefit_contribution,treated_effect_coefficient,control_effect_coefficient,avg_treated_effect_weight,avg_control_effect_weight,importance_type
0,anxiety_flag,0.002156,0.000201,0.756667,-0.000480,-0.005453,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
1,program_Complex_CM,0.001840,0.000069,0.213333,0.000749,0.004489,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
2,percolator_clinical_score,0.001645,0.000175,0.443333,-0.000091,0.005489,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
3,pcp_visits_last_6m,0.001609,-0.000381,0.533333,0.000132,-0.004960,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
4,admits_last_6m,0.001609,0.000332,0.293333,-0.001107,0.006117,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
5,ed_visits_last_6m,0.001577,0.000494,0.553333,-0.000752,0.004966,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
6,county_County_E,0.001564,-0.000125,0.806667,-0.000814,-0.003647,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
7,program_CM,0.001524,0.000034,0.476667,-0.000299,-0.003354,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
8,plan_type_Medicare,0.001476,0.000049,0.773333,-0.000486,-0.003674,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...
9,chf_flag,0.001392,0.000088,0.150000,-0.000899,0.005979,0.597458,0.402542,glmnet_xlearner_weighted_shared_standardized_e...


GLMNET X-learner benefit-driver outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/GLMNet
X-learner benefit-driver CSV: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/GLMNet/xlearner_benefit_driver_importance.csv


T-learner versus X-learner benefit-driver comparison chart saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/GLMNet/dashboard_t_vs_x_benefit_driver_comparison.png


---
## GLMNET BENEFIT SCORE SHAP CONTRIBUTIONS

This section adds SHAP-based global explanations for the final GLMNet T-learner and X-learner benefit-score functions. Mean absolute SHAP ranks global importance, while signed SHAP values show average positive and negative contribution direction.


In [41]:
def _as_feature_frame(values, feature_columns):
    if isinstance(values, pd.DataFrame):
        return values.loc[:, feature_columns]
    return pd.DataFrame(values, columns=feature_columns)


def _shap_expected_value(explanation):
    base_values = np.asarray(explanation.base_values, dtype=float)
    if base_values.ndim == 0:
        return float(base_values)
    return float(np.nanmean(base_values))


def _shap_values_frame(explanation, index, feature_columns):
    shap_array = np.asarray(explanation.values, dtype=float)
    if shap_array.ndim == 3:
        shap_array = shap_array[:, :, 0]
    return pd.DataFrame(shap_array, columns=feature_columns, index=index)


def _summarize_benefit_shap(shap_df, feature_columns, importance_type):
    positive_only = shap_df.where(shap_df > 0, 0.0)
    negative_only = shap_df.where(shap_df < 0, 0.0)

    return (
        pd.DataFrame(
            {
                "feature": feature_columns,
                "mean_abs_benefit_shap": shap_df.abs().mean(axis=0).to_numpy(),
                "mean_signed_benefit_shap": shap_df.mean(axis=0).to_numpy(),
                "mean_positive_benefit_shap": positive_only.mean(axis=0).to_numpy(),
                "mean_negative_benefit_shap": negative_only.mean(axis=0).to_numpy(),
                "pct_positive_benefit_shap": shap_df.gt(0).mean(axis=0).to_numpy(),
                "pct_negative_benefit_shap": shap_df.lt(0).mean(axis=0).to_numpy(),
                "importance_type": importance_type,
            }
        )
        .sort_values("mean_abs_benefit_shap", ascending=False)
        .reset_index(drop=True)
    )


def _save_global_benefit_shap_outputs(
    importance_df,
    shap_df,
    reconciliation_df,
    folder,
    prefix,
    model_label,
):
    importance_path = folder / f"{prefix}_global_benefit_shap_importance.csv"
    member_path = folder / f"{prefix}_member_benefit_shap_values.csv"
    reconciliation_path = folder / f"{prefix}_global_benefit_shap_reconciliation.csv"
    chart_path = folder / f"dashboard_{prefix}_global_benefit_shap.png"

    importance_df.to_csv(importance_path, index=False)
    shap_df.to_csv(member_path, index=True, index_label="row_index")
    reconciliation_df.to_csv(reconciliation_path, index=False)

    top = importance_df.head(20).sort_values("mean_abs_benefit_shap")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top["feature"], top["mean_abs_benefit_shap"])
    ax.set_title(f"{model_label}: Global SHAP Drivers of Benefit Score")
    ax.set_xlabel("Mean absolute SHAP contribution to benefit score")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    fig.savefig(chart_path, dpi=150)
    plt.close(fig)

    print(f"{model_label} global benefit SHAP outputs saved to:", folder)
    print("Importance:", importance_path)
    print("Member SHAP values:", member_path)
    print("Reconciliation:", reconciliation_path)
    print("Chart:", chart_path)


def _run_global_benefit_shap(
    benefit_predict_fn,
    background_x,
    explain_x,
    expected_benefit,
    folder,
    prefix,
    model_label,
    seed=123,
):
    if shap is None:
        print(
            f"Skipped {model_label} global benefit SHAP because the shap package is not installed. "
            "Install with: pip install shap"
        )
        return None, None, None

    feature_columns = list(explain_x.columns)
    background_sample = shap.sample(
        background_x,
        min(100, len(background_x)),
        random_state=seed,
    )

    def predict_for_shap(values):
        values_df = _as_feature_frame(values, feature_columns)
        return np.asarray(benefit_predict_fn(values_df), dtype=float)

    masker = shap.maskers.Independent(background_sample)
    explainer = shap.Explainer(
        predict_for_shap,
        masker,
        algorithm="permutation",
    )

    explanation = explainer(
        explain_x,
        max_evals=2 * len(feature_columns) + 1,
    )

    shap_df = _shap_values_frame(explanation, explain_x.index, feature_columns)

    importance_df = _summarize_benefit_shap(
        shap_df,
        feature_columns,
        f"{prefix}_final_benefit_score_permutation_shap",
    )

    baseline_value = _shap_expected_value(explanation)
    signed_sum = float(importance_df["mean_signed_benefit_shap"].sum())
    reconstructed_average = baseline_value + signed_sum
    average_predicted_benefit = float(np.mean(expected_benefit))

    reconciliation_df = pd.DataFrame(
        [
            {
                "metric": "shap_baseline_benefit",
                "value": baseline_value,
            },
            {
                "metric": "sum_mean_signed_feature_shap",
                "value": signed_sum,
            },
            {
                "metric": "baseline_plus_signed_shap_sum",
                "value": reconstructed_average,
            },
            {
                "metric": "average_predicted_benefit_from_model",
                "value": average_predicted_benefit,
            },
            {
                "metric": "reconciliation_difference",
                "value": reconstructed_average - average_predicted_benefit,
            },
        ]
    )

    _save_global_benefit_shap_outputs(
        importance_df,
        shap_df,
        reconciliation_df,
        folder,
        prefix,
        model_label,
    )

    display(importance_df.head(20))
    display(reconciliation_df)

    return importance_df, shap_df, reconciliation_df


glmnet_tlearner_global_shap_importance = None
glmnet_xlearner_global_shap_importance = None

# Use the encoded training design matrix as the SHAP background/reference population.
# Your notebook creates x_treated, x_control, and x_test, not x_train.
glmnet_shap_background_x = pd.concat([x_treated, x_control], axis=0)


if enet_treated is not None and enet_control is not None:
    treated_glmnet_pipeline = enet_treated["best_model"]
    control_glmnet_pipeline = enet_control["best_model"]

    def glmnet_tlearner_benefit_predict(frame):
        pred_treated = treated_glmnet_pipeline.predict_proba(frame)[:, 1]
        pred_control = control_glmnet_pipeline.predict_proba(frame)[:, 1]
        return pred_control - pred_treated

    glmnet_tlearner_expected_benefit = glmnet_tlearner_benefit_predict(x_test)

    (
        glmnet_tlearner_global_shap_importance,
        glmnet_tlearner_member_shap_values,
        glmnet_tlearner_shap_reconciliation,
    ) = _run_global_benefit_shap(
        glmnet_tlearner_benefit_predict,
        glmnet_shap_background_x,
        x_test,
        glmnet_tlearner_expected_benefit,
        glmnet_output_folder,
        "glmnet_tlearner",
        "GLMNET T-Learner",
        seed=123,
    )
else:
    print("Skipped GLMNET T-learner global benefit SHAP because GLMNET models were not available.")


if (
    "glmnet_tau_treated_model" in globals()
    and "glmnet_tau_control_model" in globals()
    and glmnet_tau_treated_model is not None
    and glmnet_tau_control_model is not None
):
    def glmnet_xlearner_benefit_predict(frame):
        treated_effect = glmnet_tau_treated_model.predict(frame)
        control_effect = glmnet_tau_control_model.predict(frame)
        propensity = clipped_propensity(propensity_model, frame)
        return (1 - propensity) * treated_effect + propensity * control_effect

    glmnet_xlearner_expected_benefit = glmnet_xlearner_benefit_predict(x_test)

    (
        glmnet_xlearner_global_shap_importance,
        glmnet_xlearner_member_shap_values,
        glmnet_xlearner_shap_reconciliation,
    ) = _run_global_benefit_shap(
        glmnet_xlearner_benefit_predict,
        glmnet_shap_background_x,
        x_test,
        glmnet_xlearner_expected_benefit,
        xlearner_glmnet_output_folder,
        "glmnet_xlearner",
        "GLMNET X-Learner",
        seed=123,
    )
else:
    print("Skipped GLMNET X-learner global benefit SHAP because GLMNET X-learner models were not available.")


Skipped GLMNET T-Learner global benefit SHAP because the shap package is not installed. Install with: pip install shap
Skipped GLMNET X-Learner global benefit SHAP because the shap package is not installed. Install with: pip install shap


---
## WRITE-UP MODEL EVALUATION SUMMARY
---


In [42]:
def get_group_metric(df, group, metric_col):
    if df is None:
        return np.nan
    match = df[df['group'] == group]
    if match.empty:
        return np.nan
    return float(match[metric_col].iloc[0])


def get_top_gap_metric(gap_df, metric_col):
    if gap_df is None or gap_df.empty:
        return np.nan
    match = gap_df[gap_df['uplift_decile'] == 1]
    if match.empty:
        return np.nan
    return match[metric_col].iloc[0]


def get_curve_metric(curve_summary_df, metric_col):
    if curve_summary_df is None or curve_summary_df.empty:
        return np.nan
    return float(curve_summary_df[metric_col].iloc[0])


def decile_gap_rank_correlation(gap_df):
    if gap_df is None or gap_df.empty:
        return np.nan
    valid = gap_df[['avg_predicted_benefit', 'observed_control_minus_treated_gap']].dropna()
    if len(valid) < 2:
        return np.nan
    return valid['avg_predicted_benefit'].corr(valid['observed_control_minus_treated_gap'], method='spearman')


def mean_abs_predicted_observed_gap_difference(gap_df):
    if gap_df is None or gap_df.empty:
        return np.nan
    return float(gap_df['abs_predicted_minus_observed_gap'].mean())


def build_model_evaluation_row(model_label, treated_cv_auc, control_cv_auc, treated_test_auc, control_test_auc, brier_df, calibration_summary, gap_df, curve_summary_df):
    return {
        'model': model_label,
        'treated_cv_auc': treated_cv_auc,
        'control_cv_auc': control_cv_auc,
        'treated_test_auc': treated_test_auc,
        'control_test_auc': control_test_auc,
        'treated_brier_score': get_group_metric(brier_df, 'Treated', 'brier_score'),
        'control_brier_score': get_group_metric(brier_df, 'Control', 'brier_score'),
        'treated_calibration_error': get_group_metric(calibration_summary, 'Treated', 'weighted_mean_abs_calibration_error'),
        'control_calibration_error': get_group_metric(calibration_summary, 'Control', 'weighted_mean_abs_calibration_error'),
        'top_decile_avg_predicted_benefit': get_top_gap_metric(gap_df, 'avg_predicted_benefit'),
        'top_decile_observed_control_minus_treated_gap': get_top_gap_metric(gap_df, 'observed_control_minus_treated_gap'),
        'top_decile_observed_gap_ci_lower_95': get_top_gap_metric(gap_df, 'observed_gap_ci_lower_95'),
        'top_decile_observed_gap_ci_upper_95': get_top_gap_metric(gap_df, 'observed_gap_ci_upper_95'),
        'top_decile_predicted_benefit_within_observed_gap_ci_95': get_top_gap_metric(gap_df, 'predicted_benefit_within_observed_gap_ci_95'),
        'benefit_gap_spearman_corr_by_decile': decile_gap_rank_correlation(gap_df),
        'mean_abs_predicted_minus_observed_gap_by_decile': mean_abs_predicted_observed_gap_difference(gap_df),
        'observed_gap_area_under_curve': get_curve_metric(curve_summary_df, 'observed_gap_area_under_curve'),
        'predicted_benefit_area_under_curve': get_curve_metric(curve_summary_df, 'predicted_benefit_area_under_curve'),
    }


model_evaluation_rows = [
    build_model_evaluation_row(
        'XGBoost',
        xgb_treated_cv['best_cv_auc'],
        xgb_control_cv['best_cv_auc'],
        auc_treated_cv_xgb,
        auc_control_cv_xgb,
        brier_xgboost,
        calibration_summary_xgboost,
        observed_gap_xgboost,
        uplift_curve_summary_xgboost,
    )
]

if enet_treated is not None and enet_control is not None:
    model_evaluation_rows.append(
        build_model_evaluation_row(
            'GLMNET',
            enet_treated['best_auc'],
            enet_control['best_auc'],
            auc_treated_cv_glmnet,
            auc_control_cv_glmnet,
            brier_glmnet,
            calibration_summary_glmnet,
            observed_gap_glmnet,
            uplift_curve_summary_glmnet,
        )
    )

model_evaluation_summary = pd.DataFrame(model_evaluation_rows)
model_evaluation_summary_path = output_folder / 'model_evaluation_summary.csv'
model_evaluation_summary.to_csv(model_evaluation_summary_path, index=False)

print('Model evaluation summary:')
display(model_evaluation_summary)
print('Model evaluation summary written to:', model_evaluation_summary_path)




Model evaluation summary:


,model,treated_cv_auc,control_cv_auc,treated_test_auc,control_test_auc,treated_brier_score,control_brier_score,treated_calibration_error,control_calibration_error,top_decile_avg_predicted_benefit,top_decile_observed_control_minus_treated_gap,top_decile_observed_gap_ci_lower_95,top_decile_observed_gap_ci_upper_95,top_decile_predicted_benefit_within_observed_gap_ci_95,benefit_gap_spearman_corr_by_decile,mean_abs_predicted_minus_observed_gap_by_decile,observed_gap_area_under_curve,predicted_benefit_area_under_curve
0,XGBoost,0.739937,0.667975,0.835398,0.583295,0.045232,0.068098,0.080303,0.064377,0.048251,-0.015873,-0.408481,0.269251,True,-0.533783,0.084408,-0.007789,0.014618
1,GLMNET,0.719497,0.647243,0.904425,0.659991,0.039236,0.065799,0.061494,0.051123,0.044022,-0.038278,-0.368006,0.230155,True,-0.153386,0.089042,0.020156,0.034858


Model evaluation summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/model_evaluation_summary.csv


---
## WRITE-UP MODEL RECOMMENDATION SUMMARY
---


In [43]:
def auc_strength(value):
    if pd.isna(value):
        return 'Unavailable'
    if value >= 0.90:
        return 'Excellent'
    if value >= 0.80:
        return 'Strong'
    if value >= 0.70:
        return 'Acceptable'
    if value >= 0.60:
        return 'Weak'
    return 'Poor'


def calibration_label(row):
    avg_error = np.nanmean([row['treated_calibration_error'], row['control_calibration_error']])
    avg_brier = np.nanmean([row['treated_brier_score'], row['control_brier_score']])
    if pd.isna(avg_error):
        return 'Unavailable'
    if avg_error <= 0.02:
        return f'Well calibrated; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'
    if avg_error <= 0.05:
        return f'Moderately calibrated; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'
    return f'Calibration needs review; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'


def benefit_ranking_label(row):
    benefit = row['top_decile_avg_predicted_benefit']
    gap = row['top_decile_observed_control_minus_treated_gap']
    ci_lower = row['top_decile_observed_gap_ci_lower_95']
    ci_upper = row['top_decile_observed_gap_ci_upper_95']
    benefit_within_ci = row['top_decile_predicted_benefit_within_observed_gap_ci_95']
    rank_corr = row['benefit_gap_spearman_corr_by_decile']
    if pd.isna(benefit):
        return 'Unavailable'
    ci_text = f'95% CI {ci_lower:.3f} to {ci_upper:.3f}' if pd.notna(ci_lower) else '95% CI unavailable'
    if pd.isna(benefit_within_ci):
        within_text = 'predicted benefit CI check unavailable'
    elif bool(benefit_within_ci):
        within_text = 'predicted benefit is within observed-gap CI'
    else:
        within_text = 'predicted benefit is outside observed-gap CI'
    corr_text = f'; decile rank corr {rank_corr:.3f}' if pd.notna(rank_corr) else ''
    if benefit > 0 and gap > 0:
        return f'Top decile predicted benefit {benefit:.3f}; observed gap {gap:.3f} ({ci_text}); {within_text}{corr_text}'
    if benefit > 0:
        return f'Top decile predicted benefit {benefit:.3f}; observed gap not positive ({gap:.3f}, {ci_text}); {within_text}{corr_text}'
    return f'Top decile predicted benefit is not positive ({benefit:.3f}); observed gap {gap:.3f} ({ci_text}); {within_text}{corr_text}'


recommendation_rows = []
for _, row in model_evaluation_summary.iterrows():
    avg_auc = np.nanmean([row['treated_test_auc'], row['control_test_auc']])
    recommendation_rows.append(
        {
            'model': row['model'],
            'risk_prediction_strength': f'{auc_strength(avg_auc)} average test AUC ({avg_auc:.3f})',
            'probability_calibration': calibration_label(row),
            'benefit_ranking_quality': benefit_ranking_label(row),
            'explainability': 'Tree SHAP risk and benefit drivers' if row['model'] == 'XGBoost' else 'Standardized coefficient/logit contribution risk and benefit drivers',
            'model_evaluation_takeaway': 'Prioritization candidate if AUC, calibration, observed-gap direction, and uncertainty are acceptable; ROI is evaluated separately in the business value section.',
        }
    )

model_recommendation_summary = pd.DataFrame(recommendation_rows)
model_recommendation_summary_path = output_folder / 'model_recommendation_summary.csv'
model_recommendation_summary.to_csv(model_recommendation_summary_path, index=False)

print('Model recommendation summary:')
display(model_recommendation_summary)
print('Model recommendation summary written to:', model_recommendation_summary_path)





Model recommendation summary:


,model,risk_prediction_strength,probability_calibration,benefit_ranking_quality,explainability,model_evaluation_takeaway
0,XGBoost,Acceptable average test AUC (0.709),Calibration needs review; avg abs calibration ...,Top decile predicted benefit 0.048; observed g...,Tree SHAP risk and benefit drivers,"Prioritization candidate if AUC, calibration, ..."
1,GLMNET,Acceptable average test AUC (0.782),Calibration needs review; avg abs calibration ...,Top decile predicted benefit 0.044; observed g...,Standardized coefficient/logit contribution ri...,"Prioritization candidate if AUC, calibration, ..."


Model recommendation summary written to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/model_recommendation_summary.csv


---
## RISK TIER VERSUS BENEFIT GROUP OUTPUTS
---


---
## SYNTHETIC TRUE-BENEFIT VALIDATION

Because the current dataset is synthetic, the known treatment-benefit formula can be used to validate GLMNet T-learner and GLMNet X-learner treatment-effect estimates directly.


In [44]:
def compute_synthetic_true_benefit(frame):
    return (
        0.02
        + 0.018 * frame['ed_visits_last_6m'].astype(float)
        + 0.015 * frame['admits_last_6m'].astype(float)
        + 0.018 * frame['food_insecurity_flag'].astype(float)
        + 0.014 * frame['transportation_barrier_flag'].astype(float)
        + 0.012 * frame['behavioral_health_risk_flag'].astype(float)
        + 0.0006 * np.maximum(frame['current_risk_score'].astype(float) - 50, 0)
    )


def true_benefit_validation_metrics(predicted, true):
    predicted = pd.Series(predicted, dtype=float)
    true = pd.Series(true, dtype=float)
    return {
        'mean_predicted_benefit': predicted.mean(),
        'mean_true_benefit': true.mean(),
        'bias': predicted.mean() - true.mean(),
        'mae': np.mean(np.abs(predicted - true)),
        'rmse': np.sqrt(np.mean((predicted - true) ** 2)),
        'pearson_corr': predicted.corr(true, method='pearson'),
        'spearman_corr': predicted.corr(true, method='spearman'),
    }


if results_test_glmnet is not None and 'results_test_xlearner_glmnet' in globals() and results_test_xlearner_glmnet is not None:
    glmnet_true_benefit_comparison = results_test_xlearner_glmnet[['member_id']].copy()
    glmnet_true_benefit_comparison = glmnet_true_benefit_comparison.merge(
        results_test_glmnet[['member_id', 'benefit_score']].rename(columns={'benefit_score': 'glmnet_tlearner_benefit_score'}),
        on='member_id',
        how='left',
        validate='one_to_one',
    ).merge(
        results_test_xlearner_glmnet[['member_id', 'benefit_score']].rename(columns={'benefit_score': 'glmnet_xlearner_benefit_score'}),
        on='member_id',
        how='left',
        validate='one_to_one',
    )
    xlearner_features_by_member = results_test_xlearner_glmnet.set_index('member_id')
    glmnet_true_benefit_comparison['true_benefit'] = compute_synthetic_true_benefit(
        xlearner_features_by_member.loc[glmnet_true_benefit_comparison['member_id']].reset_index(drop=True)
    ).to_numpy()
    glmnet_true_benefit_comparison['true_benefit_decile'] = ntile_desc(glmnet_true_benefit_comparison['true_benefit'], 10).to_numpy()
    glmnet_true_benefit_comparison['glmnet_tlearner_decile'] = ntile_desc(glmnet_true_benefit_comparison['glmnet_tlearner_benefit_score'], 10).to_numpy()
    glmnet_true_benefit_comparison['glmnet_xlearner_decile'] = ntile_desc(glmnet_true_benefit_comparison['glmnet_xlearner_benefit_score'], 10).to_numpy()

    true_benefit_validation_rows = []
    for model_label, score_col in [
        ('GLMNet T-learner', 'glmnet_tlearner_benefit_score'),
        ('GLMNet X-learner', 'glmnet_xlearner_benefit_score'),
    ]:
        row = {
            'model': model_label,
            'n_test_members': len(glmnet_true_benefit_comparison),
        }
        row.update(true_benefit_validation_metrics(
            glmnet_true_benefit_comparison[score_col],
            glmnet_true_benefit_comparison['true_benefit'],
        ))
        true_benefit_validation_rows.append(row)

    glmnet_true_benefit_validation_summary = pd.DataFrame(true_benefit_validation_rows)
    glmnet_true_benefit_validation_summary.to_csv(output_folder / 'glmnet_true_benefit_validation_summary.csv', index=False)
    glmnet_true_benefit_comparison.to_csv(output_folder / 'glmnet_true_benefit_scored_test_comparison.csv', index=False)
    display(glmnet_true_benefit_validation_summary)

    true_top_decile_members = set(
        glmnet_true_benefit_comparison.loc[glmnet_true_benefit_comparison['true_benefit_decile'] == 1, 'member_id']
    )
    overlap_rows = []
    for model_label, decile_col in [
        ('GLMNet T-learner', 'glmnet_tlearner_decile'),
        ('GLMNet X-learner', 'glmnet_xlearner_decile'),
    ]:
        model_top_decile_members = set(
            glmnet_true_benefit_comparison.loc[glmnet_true_benefit_comparison[decile_col] == 1, 'member_id']
        )
        overlap_members = len(model_top_decile_members & true_top_decile_members)
        overlap_rows.append({
            'model': model_label,
            'test_members': len(glmnet_true_benefit_comparison),
            'model_top_decile_members': len(model_top_decile_members),
            'true_top_decile_members': len(true_top_decile_members),
            'overlap_members': overlap_members,
            'top_decile_overlap_pct': overlap_members / len(model_top_decile_members) if model_top_decile_members else np.nan,
        })

    glmnet_true_benefit_decile_overlap_summary = pd.DataFrame(overlap_rows)
    glmnet_true_benefit_decile_overlap_summary.to_csv(output_folder / 'glmnet_true_benefit_decile_overlap_summary.csv', index=False)
    display(glmnet_true_benefit_decile_overlap_summary)
else:
    print('Skipped synthetic true-benefit validation because GLMNet T-learner or X-learner outputs are unavailable.')


,model,n_test_members,mean_predicted_benefit,mean_true_benefit,bias,mae,rmse,pearson_corr,spearman_corr
0,GLMNet T-learner,300,0.034436,0.054813,-0.020377,0.026093,0.036961,-0.467132,-0.367550
1,GLMNet X-learner,300,0.037714,0.054813,-0.017098,0.024181,0.031003,0.391002,0.168152


,model,test_members,model_top_decile_members,true_top_decile_members,overlap_members,top_decile_overlap_pct
0,GLMNet T-learner,300,30,30,2,0.066667
1,GLMNet X-learner,300,30,30,14,0.466667


In [45]:
risk_tier_order = ['Low', 'Medium', 'High', 'Very High']
benefit_group_order = ['High benefit', 'Medium benefit', 'Low benefit']

benefit_group_colors = {
    'High benefit': '#1f77b4',
    'Medium benefit': '#70ad47',
    'Low benefit': '#a6a6a6',
}

risk_tier_thresholds = pd.DataFrame(
    [
        {
            'risk_tier': 'Low',
            'current_risk_score_rule': '< 35',
            'lower_bound_inclusive': np.nan,
            'upper_bound_exclusive': 35,
        },
        {
            'risk_tier': 'Medium',
            'current_risk_score_rule': '35 to < 55',
            'lower_bound_inclusive': 35,
            'upper_bound_exclusive': 55,
        },
        {
            'risk_tier': 'High',
            'current_risk_score_rule': '55 to < 75',
            'lower_bound_inclusive': 55,
            'upper_bound_exclusive': 75,
        },
        {
            'risk_tier': 'Very High',
            'current_risk_score_rule': '>= 75',
            'lower_bound_inclusive': 75,
            'upper_bound_exclusive': np.nan,
        },
    ]
)

risk_tier_thresholds.to_csv(output_folder / 'risk_tier_thresholds.csv', index=False)

if scored_full_glmnet is None:
    raise ValueError(
        'GLMNet full scored output is unavailable. '
        'Run the GLMNet scoring cells before this risk-tier section.'
    )

if results_test_glmnet is None:
    raise ValueError(
        'GLMNet T-learner test output is unavailable. '
        'Run the GLMNet test scoring cells before this risk-tier section.'
    )

if results_test_xlearner_glmnet is None:
    raise ValueError(
        'GLMNet X-learner test output is unavailable. '
        'Run the X-learner cells before this risk-tier section.'
    )

risk_tier_population_summary = (
    scored_full_glmnet.groupby('risk_tier', observed=False)
    .agg(
        members=('risk_tier', 'size'),
        pct_population=('risk_tier', lambda s: len(s) / len(scored_full_glmnet)),
        min_current_risk_score=('current_risk_score', 'min'),
        max_current_risk_score=('current_risk_score', 'max'),
        avg_current_risk_score=('current_risk_score', 'mean'),
    )
    .reindex(risk_tier_order)
    .reset_index()
)

risk_tier_population_summary.to_csv(
    output_folder / 'risk_tier_population_summary.csv',
    index=False,
)


def assign_model_relative_benefit_group(uplift_decile):
    uplift_decile = int(uplift_decile)

    if uplift_decile in [1, 2]:
        return 'High benefit'

    if uplift_decile in [3, 4, 5, 6, 7]:
        return 'Medium benefit'

    return 'Low benefit'


def build_risk_tier_benefit_group_outputs(
    scored_df,
    output_dir,
    file_prefix,
    chart_title,
):
    df = scored_df.copy()

    df['risk_tier'] = pd.Categorical(
        df['risk_tier'],
        categories=risk_tier_order,
        ordered=True,
    )

    df['benefit_group'] = df['uplift_decile'].apply(
        assign_model_relative_benefit_group
    )

    df['benefit_group'] = pd.Categorical(
        df['benefit_group'],
        categories=benefit_group_order,
        ordered=True,
    )

    summary = (
        df.groupby(['risk_tier', 'benefit_group'], observed=False)
        .size()
        .reset_index(name='members')
    )

    tier_totals = (
        df.groupby('risk_tier', observed=False)
        .size()
        .rename('risk_tier_members')
        .reset_index()
    )

    summary = summary.merge(tier_totals, on='risk_tier', how='left')

    summary['pct_within_risk_tier'] = np.where(
        summary['risk_tier_members'] > 0,
        summary['members'] / summary['risk_tier_members'],
        np.nan,
    )

    summary['benefit_group_definition'] = summary['benefit_group'].map(
        {
            'High benefit': 'uplift_decile 1-2 (top 20%)',
            'Medium benefit': 'uplift_decile 3-7 (middle 50%)',
            'Low benefit': 'uplift_decile 8-10 (bottom 30%)',
        }
    )

    summary_path = output_dir / f'{file_prefix}_risk_tier_benefit_group_summary.csv'
    summary.to_csv(summary_path, index=False)

    pct = (
        summary.pivot(
            index='risk_tier',
            columns='benefit_group',
            values='pct_within_risk_tier',
        )
        .reindex(risk_tier_order)
        .fillna(0)
    )

    counts = (
        tier_totals.set_index('risk_tier')
        .reindex(risk_tier_order)['risk_tier_members']
        .fillna(0)
        .astype(int)
    )

    fig, ax = plt.subplots(figsize=(9.2, 6.0))

    bottom = np.zeros(len(pct))
    x = np.arange(len(pct.index))

    for group in benefit_group_order:
        values = pct[group].to_numpy() if group in pct else np.zeros(len(pct))

        ax.bar(
            x,
            values,
            bottom=bottom,
            label=group,
            color=benefit_group_colors[group],
            edgecolor='white',
            linewidth=0.8,
        )

        for idx, value in enumerate(values):
            if value >= 0.07:
                label_color = 'white' if group == 'High benefit' else '#222222'
                ax.text(
                    idx,
                    bottom[idx] + value / 2,
                    f'{value:.0%}',
                    ha='center',
                    va='center',
                    fontsize=9,
                    color=label_color,
                )

        bottom += values

    ax.set_xticks(x)

    ax.set_xticklabels(
        [f'{tier} risk\n(n={counts.loc[tier]})' for tier in risk_tier_order]
    )

    ax.set_ylim(0, 1)
    ax.set_xlabel('Risk tier based on current_risk_score', labelpad=14)
    ax.set_ylabel('Percent of members within risk tier')
    ax.set_title(chart_title)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: f'{y:.0%}')
    )

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.22),
        ncol=3,
        frameon=False,
    )

    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)

    fig.tight_layout(rect=[0, 0.08, 1, 1])

    chart_path = output_dir / f'dashboard_{file_prefix}_risk_tier_by_benefit_group.png'
    fig.savefig(chart_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

    return summary


tlearner_risk_tier_benefit_group_summary = build_risk_tier_benefit_group_outputs(
    results_test_glmnet,
    glmnet_output_folder,
    'tlearner',
    'GLMNet T-learner benefit group distribution by risk tier (test set)',
)

xlearner_risk_tier_benefit_group_summary = build_risk_tier_benefit_group_outputs(
    results_test_xlearner_glmnet,
    xlearner_glmnet_output_folder,
    'xlearner',
    'GLMNet X-learner benefit group distribution by risk tier (test set)',
)

print('Risk tier thresholds:')
display(risk_tier_thresholds)

print('Risk tier population summary:')
display(risk_tier_population_summary)

print('T-learner risk tier by benefit group summary:')
display(tlearner_risk_tier_benefit_group_summary)

print('X-learner risk tier by benefit group summary:')
display(xlearner_risk_tier_benefit_group_summary)


Risk tier thresholds:


,risk_tier,current_risk_score_rule,lower_bound_inclusive,upper_bound_exclusive
0,Low,< 35,NaN,35.0
1,Medium,35 to < 55,35.0,55.0
2,High,55 to < 75,55.0,75.0
3,Very High,>= 75,75.0,NaN


Risk tier population summary:


,risk_tier,members,pct_population,min_current_risk_score,max_current_risk_score,avg_current_risk_score
0,Low,161,0.161,14.5,34.9,30.464596
1,Medium,720,0.720,35.0,54.9,43.671806
2,High,117,0.117,55.1,73.9,60.801709
3,Very High,2,0.002,76.2,79.8,78.000000


T-learner risk tier by benefit group summary:


,risk_tier,benefit_group,members,risk_tier_members,pct_within_risk_tier,benefit_group_definition
0,Low,High benefit,4,51,0.078431,uplift_decile 1-2 (top 20%)
1,Low,Medium benefit,33,51,0.647059,uplift_decile 3-7 (middle 50%)
2,Low,Low benefit,14,51,0.274510,uplift_decile 8-10 (bottom 30%)
3,Medium,High benefit,53,207,0.256039,uplift_decile 1-2 (top 20%)
4,Medium,Medium benefit,109,207,0.526570,uplift_decile 3-7 (middle 50%)
5,Medium,Low benefit,45,207,0.217391,uplift_decile 8-10 (bottom 30%)
6,High,High benefit,3,42,0.071429,uplift_decile 1-2 (top 20%)
7,High,Medium benefit,8,42,0.190476,uplift_decile 3-7 (middle 50%)
8,High,Low benefit,31,42,0.738095,uplift_decile 8-10 (bottom 30%)
9,Very High,High benefit,0,0,NaN,uplift_decile 1-2 (top 20%)


X-learner risk tier by benefit group summary:


,risk_tier,benefit_group,members,risk_tier_members,pct_within_risk_tier,benefit_group_definition
0,Low,High benefit,3,51,0.058824,uplift_decile 1-2 (top 20%)
1,Low,Medium benefit,34,51,0.666667,uplift_decile 3-7 (middle 50%)
2,Low,Low benefit,14,51,0.274510,uplift_decile 8-10 (bottom 30%)
3,Medium,High benefit,25,207,0.120773,uplift_decile 1-2 (top 20%)
4,Medium,Medium benefit,109,207,0.526570,uplift_decile 3-7 (middle 50%)
5,Medium,Low benefit,73,207,0.352657,uplift_decile 8-10 (bottom 30%)
6,High,High benefit,32,42,0.761905,uplift_decile 1-2 (top 20%)
7,High,Medium benefit,7,42,0.166667,uplift_decile 3-7 (middle 50%)
8,High,Low benefit,3,42,0.071429,uplift_decile 8-10 (bottom 30%)
9,Very High,High benefit,0,0,NaN,uplift_decile 1-2 (top 20%)


---
## WRITE-UP OUTPUT FILE INDEX
---


In [46]:
writeup_output_files = [
    output_folder / 'data_review_summary.csv',
    predictor_distribution_folder / 'predictor_data_dictionary.csv',
    predictor_distribution_folder / 'numeric_predictor_summary.csv',
    predictor_distribution_folder / 'categorical_predictor_summary.csv',
    output_folder / 'Predictor_Distributions' / 'numeric_predictor_distributions.pdf',
    output_folder / 'Predictor_Distributions' / 'categorical_predictor_distributions.pdf',
    output_folder / 'Predictor_Distributions' / 'predictor_distribution_visual_index.csv',
    output_folder / 'model_evaluation_summary.csv',
    output_folder / 'model_recommendation_summary.csv',
    output_folder / 'factual_event_count_summary.csv',
    output_folder / 'factual_prediction_separation.csv',
    output_folder / 'factual_prediction_ranges.csv',
    output_folder / 'risk_tier_population_summary.csv',
    output_folder / 'risk_tier_thresholds.csv',
]

if 'xlearner_root_folder' in globals():
    writeup_output_files.append(xlearner_root_folder / 'xlearner_vs_tlearner_consistency_summary.csv')
    for folder in [xlearner_xgboost_output_folder, xlearner_glmnet_output_folder]:
        writeup_output_files.extend(
            [
                folder / 'xlearner_scored_test_output.csv',
                folder / 'xlearner_decile_summary.csv',
            ]
        )


    writeup_output_files.extend(
        [
            xlearner_glmnet_output_folder / 'xlearner_risk_tier_benefit_group_summary.csv',
            xlearner_glmnet_output_folder / 'dashboard_xlearner_risk_tier_by_benefit_group.png',
        ]
    )
    # XGBoost X-Learner parity outputs
    writeup_output_files.extend(
        [
            xlearner_xgboost_output_folder / 'dashboard_avg_benefit_by_decile.png',
            xlearner_xgboost_output_folder / 'xlearner_roi_by_decile.csv',
            xlearner_xgboost_output_folder / 'xlearner_risk_tier_benefit_group_summary.csv',
            xlearner_xgboost_output_folder / 'cumulative_gross_savings_by_targeting.csv',
            xlearner_xgboost_output_folder / 'marginal_gross_savings_by_targeting.csv',
            xlearner_xgboost_output_folder / 'shap_importance_benefit_score.csv',
            xlearner_xgboost_output_folder / 'xlearner_benefit_driver_importance.csv',
            xlearner_xgboost_output_folder / 'dashboard_shap_benefit_score.png',
            xlearner_xgboost_output_folder / 'dashboard_xlearner_benefit_drivers.png',
            xlearner_xgboost_output_folder / 'dashboard_xlearner_risk_tier_by_benefit_group.png',
            xlearner_xgboost_output_folder / 'dashboard_cumulative_gross_savings_targeting.png',
            xlearner_xgboost_output_folder / 'dashboard_marginal_gross_savings_advantage_vs_current_risk.png',
        ]
    )
for folder in [xgboost_output_folder, glmnet_output_folder]:
    writeup_output_files.extend(
        [
            folder / 'uplift_decile_summary.csv',
            folder / 'uplift_roi_by_decile.csv',
            folder / 'model_brier_scores.csv',
            folder / 'calibration_summary.csv',
            folder / 'calibration_by_decile.csv',
            folder / 'uplift_observed_gap_by_decile.csv',
            folder / 'uplift_curve_by_decile.csv',
            folder / 'uplift_curve_summary.csv',
            folder / 'top_benefit_decile_summary.csv',
            folder / 'shap_importance_treated_control_models.csv',
            folder / 'shap_importance_benefit_score.csv',
            folder / 'dashboard_calibration_plot.png',
            folder / 'dashboard_observed_gap_by_decile.png',
            folder / 'dashboard_uplift_curve_by_decile.png',
            folder / 'dashboard_shap_benefit_score.png',
        ]
    )


writeup_output_files.extend(
    [
        glmnet_output_folder / 'tlearner_risk_tier_benefit_group_summary.csv',
        glmnet_output_folder / 'dashboard_tlearner_risk_tier_by_benefit_group.png',
    ]
)
writeup_output_index = pd.DataFrame(
    {
        'output_file': [str(path) for path in writeup_output_files],
        'exists': [path.exists() for path in writeup_output_files],
    }
)

print('Write-up output file index:')
display(writeup_output_index)
print('\n'.join(writeup_output_index['output_file']))



Write-up output file index:


,output_file,exists
0,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
1,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
2,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
3,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
4,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
...,...,...
60,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
61,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
62,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True
63,/home/sagemaker-user/prism_repo/Outputs/Uplift...,True


/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/data_review_summary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/predictor_data_dictionary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/numeric_predictor_summary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/categorical_predictor_summary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/numeric_predictor_distributions.pdf
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/categorical_predictor_distributions.pdf
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/Predictor_Distributions/predictor_distribution_visual_index.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/model_evaluation_summary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/model_recommendation_summary.csv
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/factual_event_count_

---
## XGBOOST X-LEARNER FEATURE PARITY
---

This section brings the XGBoost X-Learner output set to parity with the GLMNet X-Learner.
It generates all model-agnostic artifacts (decile dashboards, ROI, risk-tier segmentation,
TreeSHAP benefit-driver importance, cumulative/marginal targeting savings) using shared
reporting functions.


In [47]:
# ═══════════════════════════════════════════════════════════════════════════════
# Shared X-Learner reporting functions (model-agnostic)
# ═══════════════════════════════════════════════════════════════════════════════


def generate_xlearner_decile_dashboard(decile_df, output_dir, model_label, benefit_y_max=None):
    """Generate standard decile dashboard charts for any X-Learner."""
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'avg_benefit_score',
        f'{model_label} X-Learner: Average Predicted Benefit by Uplift Decile',
        'Uplift Decile: 1 = Highest Predicted Benefit',
        'Average Benefit Score',
        output_dir / 'dashboard_avg_benefit_by_decile.png',
        y_max=benefit_y_max,
    )


def generate_xlearner_roi(decile_df, output_dir, model_label):
    """Generate ROI summary CSV and chart for any X-Learner."""
    roi_df = build_roi_summary(decile_df)
    roi_df.to_csv(output_dir / 'xlearner_roi_by_decile.csv', index=False)
    save_bar_chart(
        roi_df,
        'uplift_decile',
        'net_savings',
        f'{model_label} X-Learner: Estimated Net Savings by Uplift Decile',
        'Uplift Decile',
        'Estimated Net Savings',
        output_dir / 'dashboard_xlearner_roi_net_savings_by_decile.png',
    )
    print(f'{model_label} X-Learner ROI saved to:', output_dir)
    return roi_df


def generate_xlearner_risk_tier_outputs(results_df, output_dir, model_label):
    """Generate risk-tier by benefit-group summary for any X-Learner."""
    if 'risk_tier' not in results_df.columns or 'current_risk_score' not in results_df.columns:
        print(f'Skipped {model_label} X-Learner risk-tier outputs: required columns missing.')
        return None
    summary = build_risk_tier_benefit_group_outputs(
        results_df,
        output_dir,
        'xlearner',
        f'{model_label} X-Learner benefit group distribution by risk tier (test set)',
    )
    print(f'{model_label} X-Learner risk-tier outputs saved to:', output_dir)
    return summary


def generate_xlearner_cumulative_targeting(results_df, output_dir, model_label):
    """Generate cumulative and marginal gross savings targeting analysis."""
    n_total = len(results_df)
    if n_total == 0:
        return None, None, None
    decile_size = max(1, n_total // 10)

    ranking_specs = [
        ('Uplift score', 'benefit_score', False),
    ]
    if 'current_risk_score' in results_df.columns:
        ranking_specs.append(('Current risk score', 'current_risk_score', False))

    rows = []
    for approach, sort_col, ascending in ranking_specs:
        ranked = results_df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
        for decile in range(1, 11):
            top_n = n_total if decile == 10 else decile * decile_size
            selected = ranked.head(top_n)
            avoided = float(selected['benefit_score'].sum())
            rows.append({
                'targeting_approach': approach,
                'through_decile': decile,
                'population_fraction_targeted': top_n / n_total,
                'n': top_n,
                'cumulative_estimated_ed_visits_avoided': avoided,
                'cumulative_gross_savings': avoided * cost_per_ed_visit,
            })

    cumulative_df = pd.DataFrame(rows)
    cumulative_df.to_csv(output_dir / 'cumulative_gross_savings_by_targeting.csv', index=False)

    # Summary at top 50%
    summary_rows = []
    for approach in cumulative_df['targeting_approach'].unique():
        subset = cumulative_df[cumulative_df['targeting_approach'] == approach]
        top50 = subset[subset['population_fraction_targeted'] <= 0.51].iloc[-1]
        summary_rows.append({
            'targeting_approach': approach,
            'population_fraction_targeted': top50['population_fraction_targeted'],
            'n': int(top50['n']),
            'cumulative_gross_savings': top50['cumulative_gross_savings'],
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(output_dir / 'cumulative_gross_savings_summary_top50.csv', index=False)

    # Marginal savings by targeting approach
    marginal_rows = []
    for approach in cumulative_df['targeting_approach'].unique():
        subset = cumulative_df[cumulative_df['targeting_approach'] == approach].sort_values('through_decile')
        prev = 0.0
        for _, row in subset.iterrows():
            marginal = row['cumulative_gross_savings'] - prev
            marginal_rows.append({
                'targeting_approach': approach,
                'decile': int(row['through_decile']),
                'marginal_gross_savings': marginal,
                'cumulative_gross_savings': row['cumulative_gross_savings'],
            })
            prev = row['cumulative_gross_savings']
    marginal_df = pd.DataFrame(marginal_rows)
    marginal_df.to_csv(output_dir / 'marginal_gross_savings_by_targeting.csv', index=False)

    # Marginal advantage vs current risk
    if 'Current risk score' in cumulative_df['targeting_approach'].values:
        uplift_marginals = marginal_df[marginal_df['targeting_approach'] == 'Uplift score'].set_index('decile')['marginal_gross_savings']
        risk_marginals = marginal_df[marginal_df['targeting_approach'] == 'Current risk score'].set_index('decile')['marginal_gross_savings']
        advantage_rows = []
        for decile in range(1, 11):
            u = uplift_marginals.get(decile, 0)
            r = risk_marginals.get(decile, 0)
            advantage_rows.append({
                'decile': decile,
                'uplift_marginal_gross_savings': u,
                'risk_marginal_gross_savings': r,
                'marginal_advantage': u - r,
            })
        advantage_df = pd.DataFrame(advantage_rows)
        advantage_df.to_csv(output_dir / 'marginal_gross_savings_advantage_vs_current_risk.csv', index=False)
    else:
        advantage_df = None

    # Cumulative savings chart
    chart_df = cumulative_df[cumulative_df['population_fraction_targeted'] <= 0.51]
    fig, ax = plt.subplots(figsize=(8.5, 5.25))
    for approach in chart_df['targeting_approach'].unique():
        approach_df = chart_df[chart_df['targeting_approach'] == approach]
        ax.plot(
            approach_df['population_fraction_targeted'],
            approach_df['cumulative_gross_savings'],
            marker='o',
            label=approach,
        )
    ax.set_title(f'{model_label} X-Learner: Cumulative Gross Savings by Targeting Approach')
    ax.set_xlabel('Population Fraction Targeted')
    ax.set_ylabel('Cumulative Gross Savings ($)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_cumulative_gross_savings_targeting.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    # Marginal advantage chart
    if advantage_df is not None:
        fig, ax = plt.subplots(figsize=(8.5, 5.25))
        ax.bar(
            advantage_df['decile'].astype(str),
            advantage_df['marginal_advantage'],
            color='#4c78a8',
        )
        ax.axhline(0, color='gray', linewidth=1)
        ax.set_title(f'{model_label} X-Learner: Marginal Gross Savings Advantage vs Current Risk')
        ax.set_xlabel('Decile')
        ax.set_ylabel('Marginal Gross Savings Advantage ($)')
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        fig.savefig(output_dir / 'dashboard_marginal_gross_savings_advantage_vs_current_risk.png', dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'{model_label} X-Learner targeting/savings outputs saved to:', output_dir)
    return cumulative_df, marginal_df, advantage_df


def generate_xlearner_benefit_shap_xgboost(tau_treated_model, tau_control_model, propensity_scores, x_matrix, output_dir, model_label):
    """Generate TreeSHAP benefit-driver outputs for XGBoost X-Learner effect models."""
    # Get TreeSHAP contributions from both effect models
    dmatrix = make_dmatrix(x_matrix)
    tau_treated_contribs = tau_treated_model.predict(dmatrix, pred_contribs=True)
    tau_control_contribs = tau_control_model.predict(dmatrix, pred_contribs=True)

    shap_columns = [*x_matrix.columns, 'BIAS']
    tau_treated_shap = pd.DataFrame(tau_treated_contribs, columns=shap_columns, index=x_matrix.index)
    tau_control_shap = pd.DataFrame(tau_control_contribs, columns=shap_columns, index=x_matrix.index)

    # Weighted combination: benefit = (1 - e(x)) * tau1(x) + e(x) * tau0(x)
    treated_weight = 1 - propensity_scores
    control_weight = propensity_scores

    weighted_treated_shap = tau_treated_shap.mul(treated_weight, axis=0)
    weighted_control_shap = tau_control_shap.mul(control_weight, axis=0)
    benefit_shap = weighted_treated_shap + weighted_control_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')

    # Importance summary
    importance_df = pd.DataFrame({
        'feature': benefit_shap_no_bias.columns,
        'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
        'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
        'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
        'importance_type': 'xgboost_xlearner_weighted_treeshap',
    }).sort_values('mean_abs_benefit_shap', ascending=False).reset_index(drop=True)

    # Save outputs
    importance_df.to_csv(output_dir / 'shap_importance_benefit_score.csv', index=False)
    importance_df.to_csv(output_dir / 'xlearner_benefit_driver_importance.csv', index=False)
    benefit_shap_no_bias.to_csv(output_dir / 'xgboost_xlearner_member_benefit_shap_values.csv', index=True, index_label='row_index')

    # Reconciliation
    baseline_treated = float(tau_treated_shap['BIAS'].mean()) if 'BIAS' in tau_treated_shap.columns else 0.0
    baseline_control = float(tau_control_shap['BIAS'].mean()) if 'BIAS' in tau_control_shap.columns else 0.0
    avg_treated_weight = float(np.mean(treated_weight))
    avg_control_weight = float(np.mean(control_weight))
    weighted_baseline = baseline_treated * avg_treated_weight + baseline_control * avg_control_weight
    signed_sum = float(importance_df['mean_signed_benefit_shap'].sum())

    reconciliation_df = pd.DataFrame([
        {'metric': 'weighted_shap_baseline', 'value': weighted_baseline},
        {'metric': 'sum_mean_signed_feature_shap', 'value': signed_sum},
        {'metric': 'baseline_plus_signed_shap_sum', 'value': weighted_baseline + signed_sum},
    ])
    reconciliation_df.to_csv(output_dir / 'xgboost_xlearner_global_benefit_shap_reconciliation.csv', index=False)

    # Global benefit SHAP importance chart
    top = importance_df.head(20).sort_values('mean_abs_benefit_shap')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_benefit_shap'])
    ax.set_title(f'{model_label} X-Learner: Top Drivers of Predicted Treatment Benefit (TreeSHAP)')
    ax.set_xlabel('Mean Absolute SHAP Contribution to Benefit')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_shap_benefit_score.png', dpi=150)
    fig.savefig(output_dir / 'dashboard_xlearner_benefit_drivers.png', dpi=150)
    plt.close(fig)

    # Global benefit SHAP chart (matching GLMNet naming pattern)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_benefit_shap'])
    ax.set_title(f'{model_label} X-Learner: Global SHAP Drivers of Benefit Score')
    ax.set_xlabel('Mean absolute SHAP contribution to benefit score')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(output_dir / 'dashboard_xgboost_xlearner_global_benefit_shap.png', dpi=150)
    plt.close(fig)

    print(f'{model_label} X-Learner TreeSHAP benefit-driver outputs saved to:', output_dir)
    return importance_df, benefit_shap_no_bias, reconciliation_df


# ═══════════════════════════════════════════════════════════════════════════════
# Generate XGBoost X-Learner outputs using shared functions
# ═══════════════════════════════════════════════════════════════════════════════

# 1. Decile dashboard chart
generate_xlearner_decile_dashboard(
    decile_summary_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 2. ROI by decile
xgboost_xlearner_roi = generate_xlearner_roi(
    decile_summary_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 3. Risk tier by benefit group
xgboost_xlearner_risk_tier_summary = generate_xlearner_risk_tier_outputs(
    results_test_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 4. Cumulative and marginal targeting savings
(
    xgboost_xlearner_cumulative,
    xgboost_xlearner_marginal,
    xgboost_xlearner_advantage,
) = generate_xlearner_cumulative_targeting(
    results_test_xlearner_xgboost,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

# 5. TreeSHAP benefit-driver importance
(
    xgboost_xlearner_shap_importance,
    xgboost_xlearner_member_shap,
    xgboost_xlearner_shap_reconciliation,
) = generate_xlearner_benefit_shap_xgboost(
    xgb_tau_treated_model,
    xgb_tau_control_model,
    propensity_test,
    x_test,
    xlearner_xgboost_output_folder,
    'XGBoost',
)

print()
print('XGBoost X-Learner feature parity outputs complete.')
print('Files in XGBoost X-Learner folder:')
print('\\n'.join(str(p) for p in sorted(xlearner_xgboost_output_folder.iterdir())))


XGBoost X-Learner ROI saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost


XGBoost X-Learner risk-tier outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost


XGBoost X-Learner targeting/savings outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost


XGBoost X-Learner TreeSHAP benefit-driver outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost

XGBoost X-Learner feature parity outputs complete.
Files in XGBoost X-Learner folder:
/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/cumulative_gross_savings_by_targeting.csv\n/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/cumulative_gross_savings_summary_top50.csv\n/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/dashboard_avg_benefit_by_decile.png\n/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/dashboard_cumulative_gross_savings_targeting.png\n/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/dashboard_marginal_gross_savings_advantage_vs_current_risk.png\n/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/dashboard_shap_benefit_score.png\n/home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/XGBoost/dashb

---
## REGENERATE GLMNET X-LEARNER OUTPUTS VIA SHARED FUNCTIONS
---

Use the same shared reporting functions to regenerate the GLMNet X-Learner outputs,
confirming functional equivalence and consolidating duplicate logic.


In [48]:
# Regenerate GLMNet X-Learner outputs through the same shared functions.
# The GLMNet X-Learner already has these files, but regenerating confirms parity.

if results_test_xlearner_glmnet is not None and decile_summary_xlearner_glmnet is not None:
    # Decile dashboard
    generate_xlearner_decile_dashboard(
        decile_summary_xlearner_glmnet,
        xlearner_glmnet_output_folder,
        'GLMNet',
    )

    # Cumulative/marginal savings (regenerate using shared function)
    generate_xlearner_cumulative_targeting(
        results_test_xlearner_glmnet,
        xlearner_glmnet_output_folder,
        'GLMNet',
    )

    print()
    print('GLMNet X-Learner outputs regenerated via shared functions.')
else:
    print('Skipped GLMNet X-Learner regeneration: models not available.')


GLMNet X-Learner targeting/savings outputs saved to: /home/sagemaker-user/prism_repo/Outputs/Uplift/Python/X-Learner/GLMNet

GLMNet X-Learner outputs regenerated via shared functions.


In [ ]:
# ============================================================================
# Regenerate README tables/charts from the outputs just produced
# ============================================================================
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "Code" / "generate_all_readmes.py")],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)
